# Cara LEGO con FLUX.2-klein-4B + ensamblado LDraw

Foto de una persona → estampa de cara RGBA (FLUX.2-klein-4B, sin entrenar) → minifig en LDraw (`.ldr`) con la cara
aplicada por `!TEXMAP`. El render final se hace fuera de Kaggle (LDView o Blender; instrucciones en el README que se genera).

**Antes de correr (panel derecho de Kaggle):**
1. *Add Input* → **PairLegoFace** y **LegosPartsVae** (las caras LEGO reales: de ahí salen el tamaño y los márgenes).
2. *Settings* → **Internet: On** y **Accelerator: GPU T4 x2**.
3. *Run All*. La primera vez descarga FLUX (~16 GB) y la librería LDraw (~140 MB).
4. Al terminar, descarga **un solo archivo**: `flux_cara/entregar.zip` (panel *Output*). Trae el reporte con todo lo
   impreso, las grillas, `tabla_prompts.csv`, las caras RGBA y la carpeta LDraw lista para renderizar.
   Si algo falla, el zip se arma igual con el error en `reporte.txt`: mándalo tal cual.

| Etapa | Qué hace | Salidas |
|---|---|---|
| Datos | 10 fotos de prueba con variedad + márgenes de las estampas reales | `datos/` |
| 1 | Cargar FLUX una vez y generar 1 imagen por foto | `etapa1/grilla_etapa1.jpg`, `prompts.csv`, `tiempos.csv` |
| 2 | Detectar la falla más común, probar 3 variantes del prompt y congelar la mejor | `etapa2/tabla_prompts.csv`, `prompt_congelado.txt`, grillas |
| 3 | Post-proceso a RGBA con guardrail | `etapa3/grilla_rgba.jpg`, `caras/*.png` |
| 4 | `generar_cara(foto, rasgos, n=5)` con reintentos | `etapa4/` |
| 5 | Minifig LDraw + README de render + vista previa | `etapa5/<foto>/` |
| 6 | Anexo A: misma foto con modelo propio y FLUX · ronda 1, ronda 2 (con un cambio) y Variar · 5 caras cada una | `etapa6/mosaico_rondas.jpg`, `rondas.csv` |

Todos los parámetros están en **`config.py`** (celda siguiente).

In [ ]:
# Instalación (solo en Kaggle). FLUX.2-klein necesita diffusers >= 0.37 (Flux2KleinPipeline) y transformers con Qwen3.
# Se mira la versión SIN importar diffusers, para no dejar una versión vieja cargada en memoria.
import os, sys, platform, subprocess
from importlib.metadata import version, PackageNotFoundError
PAQUETES = ["diffusers>=0.37.0", "transformers>=4.51.0", "accelerate"]
def _ver(p):
    try:
        return tuple(int(x) for x in version(p).split(".")[:2] if x.isdigit())
    except PackageNotFoundError:
        return (0, 0)
if os.path.isdir("/kaggle/working") and (_ver("diffusers") < (0, 37) or _ver("transformers") < (4, 51)):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PAQUETES], check=True)
for p in ("diffusers", "transformers", "accelerate", "torch"):
    try:
        print(p, version(p))
    except PackageNotFoundError:
        print(p, "(no instalado)")
os.makedirs("/kaggle/working/flux_cara/src", exist_ok=True)
print("Python", platform.python_version())

## Parámetros (`config.py`)
Todo valor incierto está aquí. Si cambias algo, vuelve a correr esta celda y la de *Importar módulos*.

In [ ]:
%%writefile /kaggle/working/flux_cara/src/config.py
"""Parámetros del proyecto. Todo valor incierto vive aquí, no escondido en los módulos.

En Kaggle este archivo lo escribe la primera celda del notebook: edítalo ahí y vuelve a correr.
"""
import os
from pathlib import Path

# ----------------------------------------------------------------- rutas (Kaggle)
ENTRADA = Path(os.environ.get("FLUX_CARA_ENTRADA", "/kaggle/input"))
SALIDA = Path(os.environ.get("FLUX_CARA_SALIDA", "/kaggle/working/flux_cara"))
RUTA_DATASET = None                  # None = buscar PairLegoFace bajo ENTRADA; o la ruta exacta
PATRON_NOMBRE_DATASET = r"pair.?lego.?face"

# ----------------------------------------------------------------- Etapa 0: exploración
MUESTRA_TAMANOS = 300                # imágenes por carpeta a las que se les lee el tamaño (None = todas)
LINEAS_ARBOL = 60                    # máximo de carpetas que imprime el árbol

# Cómo se reconocen las columnas de las tablas (regex, sin distinguir mayúsculas). El orden importa:
# una columna queda asignada al primer rol que la reconoce.
PATRONES_COLUMNA = {
    "particion": r"split|partic|subset|^set$|conjunto",
    "estampa": r"lego|mascara|máscara|mask|estampa|stamp|decal",
    "foto": r"foto|photo|imagen|image|img|ruta|path|file|archivo",
    "id": r"^id$|^id_|_id$",
    "version": r"^version$|^versión$|^g$",
    "ok": r"^pasa$|^ok$|^valid[ao]?$|^aprobad",
    "color_pelo": r"(colou?r.*(pelo|cabell|hair|ceja|brow))|((pelo|cabell|hair|ceja|brow).*colou?r)",
    "cabello": r"cabell|pelo|hair|peinado",
    "lentes": r"lente|glass|gafa|anteojo",
    "barba": r"barba|beard",
    "bigote": r"bigote|mous|mustach",
    "vello": r"vello|facial",
    "expresion": r"expres|emoci|gesto|smile|sonri",
    "femenino": r"femenin|^mujer|female",
    "edad": r"^edad$|^age$",
    "texto": r"prompt|descrip|caption",        # texto libre del que se pueden leer expresión y cabello
}
# Palabras en la ruta (relativa al dataset, sin el nombre del dataset) que delatan una estampa
PALABRAS_ESTAMPA = r"lego|mascara|máscara|mask|estampa|stamp|decal"
# Las estampas se buscan por nombre en TODO ENTRADA (pueden venir en otro Input de Kaggle)
BUSCAR_ESTAMPAS_EN_TODO_INPUT = True
LADO_MAX_ESTAMPA = 256               # una imagen con alfa y lado <= esto se considera estampa
PARTICIONES = {                      # nombre normalizado <- palabras en la columna o en las carpetas
    "prueba": ("test", "prueba", "testing"),
    "validacion": ("val", "valid", "validation", "validacion", "validación"),
    "entrenamiento": ("train", "training", "entrenamiento"),
}
VALORES_SI = {"true", "1", "1.0", "si", "sí", "yes", "y", "x", "t"}
VALORES_NO = {"false", "0", "0.0", "no", "n", "f", "ninguno", "ninguna", "none"}

# Expresión del dataset -> una de: sonrisa, serio, sorpresa, neutral.
MAPEO_EXPRESION = {}                 # p. ej. {"sonrie_dientes": "sonrisa"}; manda sobre los patrones
PATRONES_EXPRESION = [               # se prueban en orden si el valor no está en MAPEO_EXPRESION
    ("sorpresa", r"sorpr|surpri|asombr|open"),
    ("neutral", r"neutr|calm"),
    ("sonrisa", r"sonr|smil|feliz|happy|ri[eé]|alegr"),
    ("serio", r"seri|enoj|angry|triste|sad"),
]
# Si la tabla no tiene columna de expresión/cabello, se leen del texto libre (columna "texto", p. ej. el prompt).
# Se prueban en orden; el primero que calza gana.
PATRONES_EXPRESION_TEXTO = [
    ("serio", r"not smil|no smil|without (a )?smil|serious|stern|frown|scowl|angry|grumpy|unsmiling|grimac|annoyed"
              r"|worried|determined|pursed|closed mouth|straight mouth|sad|serio"),
    ("sorpresa", r"surpris|shock|astonish|amazed|scream|gasp|o-shaped mouth"
                 r"|(open[- ]mouth(ed)?|mouth open)(?![\w -]{0,15}(smil|grin|laugh))|sorpr"),
    ("sonrisa", r"smil|smirk|grin|laugh|joyful|cheerful|teeth|tongue|sonr"),
    ("neutral", r"neutral|calm|relaxed|expressionless|blank expression"),
]
# En PairLegoFace el prompt describe la cara LEGO de origen: "...facial features from the LEGO face: white thick
# eyebrows, lopsided grin. Keep the same expression...". Si este patrón calza, expresión y cejas se leen SOLO de
# esa cláusula (el resto del prompt habla de "expression" y "facial hair" en general y confunde).
PATRON_CLAUSULA_RASGOS = r"features from the lego face:\s*(.+?)\.(?:\s|$)"
# Color de cejas: se busca el color dentro del trozo de la cláusula que dice "eyebrows" (el más largo primero)
COLORES_CEJAS = ["light bluish gray", "dark bluish gray", "reddish brown", "medium nougat", "dark brown",
                 "light brown", "dark orange", "dark tan", "nougat", "auburn", "blonde", "brown", "black", "white",
                 "gray", "grey", "orange", "ginger", "red", "tan", "yellow", "silver"]
MAPEO_COLOR_CEJAS = {"nougat": "light brown", "medium nougat": "light brown", "dark tan": "tan",
                     "light bluish gray": "gray", "dark bluish gray": "dark gray", "grey": "gray", "silver": "gray"}
PATRONES_CABELLO_TEXTO = [
    ("calvo", r"bald|shaved head|calv"),
    ("largo", r"long (\w+ ){0,2}hair|ponytail|braid|pelo largo"),
    ("rizado", r"curly|afro|rizad"),
    ("corto", r"short (\w+ ){0,2}hair|buzz cut|crew cut|pelo corto"),
]
PATRON_COLOR_PELO_TEXTO = r"\b(black|dark brown|light brown|brown|blonde?|red|ginger|auburn|gr[ae]y|white|silver)\b[\w ,-]{0,15}?\bhair"

# Rasgos a mano: mandan sobre los del dataset. Clave = nombre de archivo de la foto (o su id).
# Ej.: {"p00_00.jpg": {"lentes": True, "barba": False, "bigote": False, "expresion": "sonrisa",
#                      "cabello": "corto", "color_pelo": "black"}}
RASGOS_MANUALES = {}

# Estampas sin alfa: el trazo se separa del fondo como en el post-proceso
BORDE_FONDO = 0.05                   # 5 % exterior para estimar el fondo
UMBRAL_LAB_ESTAMPA = 25.0            # distancia Lab al fondo para ser trazo

# Tamaño final de la estampa (ancho, alto). None = el más común entre las estampas reales.
TAMANO_FINAL = None
TAMANO_POR_DEFECTO = (128, 118)      # (ancho, alto) medido en las 351 estampas de LegosPartsVae (100 %)
MARGEN_POR_DEFECTO = dict(arriba=0.10, abajo=0.10, izquierda=0.10, derecha=0.10)

# Selección de las 10 fotos de prueba
N_FOTOS = 10
SEMILLA = 1234
VARIEDAD = [("lentes", True), ("lentes", False), ("barba", True), ("bigote", True),
            ("expresion", "sonrisa"), ("expresion", "serio"),
            ("expresion", "sorpresa"), ("expresion", "neutral"),
            ("femenino", True), ("femenino", False)]
NOVEDAD = ("pieza", "edad")          # bonus por elegir una cara LEGO o una edad que aún no está entre las elegidas
FOTOS_FORZADAS = []                  # nombres de archivo que quieres sí o sí entre las 10

# ----------------------------------------------------------------- Etapa 1: FLUX.2-klein-4B
# Verificado en la ficha oficial (huggingface.co/black-forest-labs/FLUX.2-klein-4B, Apache-2.0):
#   model_index.json -> Flux2KleinPipeline (transformer Flux2Transformer2DModel, text_encoder Qwen3ForCausalLM),
#   "is_distilled": true; ejemplo oficial con guidance_scale=1.0 y num_inference_steps=4; la imagen de
#   referencia se pasa con image= (PIL o lista de PIL). Requiere diffusers >= 0.37.
# device_map="balanced": soportado a nivel de pipeline en diffusers (experimental; solo "balanced").
MODELO_ID = "black-forest-labs/FLUX.2-klein-4B"
DTYPE = "bfloat16"                   # el del ejemplo oficial. En T4 (sin bf16 nativo) va más lento; "float16" puede dar NaN
DEVICE_MAP = "balanced"              # reparte los componentes entre las 2 T4
MAX_MEMORY = None                    # p. ej. {0: "14GiB", 1: "14GiB"}; None = toda la memoria de cada GPU
PASOS = 4
GUIA = 1.0
ANCHO, ALTO = 512, 512
SEMILLA_FLUX = 42                    # semilla fija (igual para las 10 fotos)
GENERADOR_EN = "cpu"                 # torch.Generator en cpu: reproducible aunque el pipeline esté repartido
LADO_REFERENCIA = 512                # la foto de referencia se recorta al centro a este cuadrado

PROMPT_BASE = (
    "Flat 2D printed face decal of a LEGO minifigure, front view, on a plain solid bright yellow background "
    "that fills the whole image. It is the LEGO face of the person in the reference photo: {expresion}, "
    "{lentes}, {vello}, eyebrows in {color}. Simple printed strokes with sharp edges and flat solid colors, "
    "small black oval eyes, only the facial features. No head outline, no hair, no shading, no 3D, no shadows, "
    "no text, no realistic skin."
)
FRASES = {                           # tabla de rasgos -> frase (de la especificación)
    "lentes": {True: "wearing black rectangular glasses", False: "no glasses"},
    "barba": "short beard",
    "bigote": "moustache",
    "expresion": {"sonrisa": "a wide smile", "serio": "a closed straight mouth and flat eyebrows",
                  "sorpresa": "an open round mouth and raised eyebrows", "neutral": "a small calm smile"},
}
FRASE_SIN_VELLO = "no facial hair"   # supuesto: sin barba ni bigote
UNIR_VELLO = " and "                 # barba + bigote -> "short beard and moustache"
EXPRESION_POR_DEFECTO = "neutral"    # si la foto no tiene expresión (queda marcado en prompts.csv)
COLOR_CEJAS_POR_DEFECTO = "black"    # si no se leyó el color de cejas

# ----------------------------------------------------------------- Etapa 2: ajuste del prompt
# Juez automático de cada imagen (estimación; la grilla de cada variante queda para revisar a ojo).
USAR_CLIP = True                     # CLIP zero-shot para "¿es plana?", "¿tiene lentes/barba?", "¿solo rasgos?"
MODELO_CLIP = "openai/clip-vit-base-patch32"
CLIP_TEXTOS = {
    "plana": ["a flat 2D vector drawing of a cartoon face", "a 3D render of a toy head", "a realistic photo of a person"],
    "lentes": ["a cartoon face wearing glasses", "a cartoon face without glasses"],
    "barba": ["a cartoon face with a beard", "a cartoon face without a beard"],
    "solo_rasgos": ["eyes, eyebrows and mouth printed on a plain yellow background",
                    "a complete cartoon head with a face outline, ears and hair"],
    # Etapa 5 (elegir la cara de la minifig): "moustache" solo no separa; "facial hair" sí (corrida 2026-11-16:
    # con bigote 0.63–0.82, sin bigote 0.31–0.41, sin vello <= 0.35, con barba >= 0.89)
    "vello": ["a cartoon face with facial hair", "a clean-shaven cartoon face"],
}
UMBRAL_RASGO_CLIP = 0.5              # prob. del primer texto para decir "tiene lentes" / "tiene vello"
AMARILLO_COMPUESTO = (250, 215, 0)   # fondo sobre el que se juzga la estampa RGBA
# Fondo amarillo: mediana del borde (BORDE_FONDO) en HSV + que la mayoría del borde sea ese fondo
AMARILLO_H = (35, 75)                # tono en grados
AMARILLO_S_MIN, AMARILLO_V_MIN = 0.45, 0.55
# Plana (respaldo si no hay CLIP): fracción de píxeles del trazo con degradé suave (sombreado) en L*
GRADIENTE_SUAVE = (2.0, 12.0)
MAX_FRACCION_SUAVE = 0.20
REVISION_MANUAL = {}                 # {"v1_vector": {3: {"plana": False}}} manda sobre el juez automático
META_PLANAS_AMARILLAS = 8            # de 10
# 3 variantes por falla: la Etapa 2 detecta la falla más común de la Etapa 1 y prueba las 3 de esa falla.
VARIANTES = {
    "3d": {
        "v1_vector": "Flat vector illustration, like a printed sticker with zero depth. " + PROMPT_BASE
                     + " Pure 2D graphic: no gradients, no lighting, no volume.",
        "v2_tinta": "A simple black ink print on a flat bright yellow surface, seen straight on: the LEGO minifigure face "
                    "of the person in the reference photo, drawn as minimal graphic strokes: {expresion}, {lentes}, "
                    "{vello}, eyebrows in {color}, small black oval eyes. Flat solid colors only. No head shape, no hair, "
                    "no shading, no 3D, no shadows, no text, no realistic skin.",
        "v3_corto": "2D flat printed LEGO face decal on solid bright yellow: {expresion}, {lentes}, {vello}, eyebrows "
                    "in {color}, small black oval eyes. Flat colors, sharp edges. Not 3D, no shading, no hair, no head outline.",
    },
    "fondo": {
        "v1_fondo_hex": PROMPT_BASE + " The background is one flat uniform bright yellow color (#F2CD37) from edge to "
                        "edge, with nothing else in it.",
        "v2_fondo_primero": "Solid bright yellow background filling the entire image, edge to edge. On it: " + PROMPT_BASE,
        "v3_fondo_papel": "A flat sheet of bright yellow paper filling the whole frame, with a LEGO minifigure face printed "
                          "on it, based on the person in the reference photo: {expresion}, {lentes}, {vello}, eyebrows "
                          "in {color}, small black oval eyes. Flat solid colors, sharp edges. No head outline, no hair, "
                          "no shading, no 3D, no shadows, no text.",
    },
    "cara_completa": {
        "v1_solo_rasgos": PROMPT_BASE + " Draw ONLY the eyes, eyebrows, mouth (and glasses or facial hair if mentioned) "
                          "directly on the yellow background; do not draw the head shape, face contour, ears, neck or hair.",
        "v2_lista": "Bright solid yellow background. Printed on it, nothing but these facial marks of the person in the "
                    "reference photo: two small black oval eyes, eyebrows in {color}, {expresion}, {lentes}, {vello}. "
                    "No face outline, no head, no ears, no hair, no skin, no shading, no 3D.",
        "v3_recortado": "Close-up of a flat LEGO face print, only the printed features, cropped tight on a plain "
                        "bright yellow background: {expresion}, {lentes}, {vello}, eyebrows in {color}, small "
                        "black oval eyes. No head outline, no hair, no shading, no 3D, no shadows, no text.",
    },
}

# ----------------------------------------------------------------- Etapa 3: post-proceso a RGBA
UMBRAL_LAB = 25.0                    # distancia Lab al fondo para ser trazo
APERTURA = 3                         # apertura morfológica 3x3
MIN_COMPONENTE = 15                  # se borran componentes de menos de 15 px
ESTADISTICO_MARGEN = "mediana"       # "mediana" o "media" de las estampas reales (margenes.json)
MARGEN_LATERAL_MIN = 0.05            # el trazo se centra sin deformarlo; nunca más cerca del borde que esto
COBERTURA_MIN, COBERTURA_MAX = 0.02, 0.50   # guardrail: fracción del área que es trazo
BORDE_FONDO_MIN = 0.60               # guardrail: fracción del borde que debe ser fondo
# guardrail "cabeza dibujada": área de la silueta rellena más grande. Medido en la corrida del 2026-11-16:
# cabezas (contorno o piel) 0.39–0.51 · solo rasgos (incl. lentes y barba en U) <= 0.18
SILUETA_MAX = 0.30
SILUETA_CIERRE = 5                   # iteraciones de cierre 3x3 para tapar cortes chicos del contorno

# ----------------------------------------------------------------- Etapa 4: generación completa
N_CARAS = 5
SALTO_REINTENTO, MAX_REINTENTOS = 1000, 2   # semilla + 1000, máximo 2 reintentos
N_FOTOS_ETAPA4 = 3                   # cuántas de las 10 fotos pasan por la Etapa 4 (5 caras cada una)

# ----------------------------------------------------------------- Etapa 6: mosaico del Anexo A
# Ronda = una llamada que genera las N_CARAS caras de una foto, como en la app.
#   Ronda 1: rasgos de la foto, semillas SEMILLA_FLUX + k (las de la Etapa 4, se reutilizan)
#   Ronda 2: el usuario pide un cambio (CAMBIO_RONDA2), mismas semillas -> la diferencia es solo el cambio
#   Variar:  mismo pedido que la ronda 2, semillas SEMILLA_FLUX + SALTO_VARIAR + k
N_PERSONAS_MOSAICO = 3               # personas distintas (las primeras de las 10 elegidas); el Anexo A usa una
# Cambio pedido en la ronda 2. "alternar" invierte un rasgo sí/no (lentes, barba, bigote).
# También vale un valor fijo: {"expresion": "sorpresa"}, {"barba": True}, {"color_cejas": "brown"}.
CAMBIO_RONDA2 = {"lentes": "alternar"}
CAMBIOS_RONDA2_MANUAL = {}           # por foto, manda sobre CAMBIO_RONDA2: {"3626cp76_04.jpg": {"expresion": "sorpresa"}}
SALTO_VARIAR = 100                   # Variar -> semillas 142..146 (no debe chocar con SALTO_REINTENTO = 1000)
NOMBRE_PROPIO = "Modelo propio"      # etiqueta de las filas del modelo del equipo
CARPETA_PROPIO = None                # p. ej. "/kaggle/input/caras-propio": <foto>/{ronda1,ronda2,variar}/*.png

# ----------------------------------------------------------------- Etapa 5: LDraw
# Verificado en la librería oficial (complete.zip de library.ldraw.org):
#   3626c.dat "Minifig Head with Hollow Stud" = s\3626cs01.dat + 5 líneas de la mitad frontal;
#   3626cpx0.dat (cabeza oficial con textura) aplica la cara con
#   "0 !TEXMAP START PLANAR -13 0 10 13 0 10 -13 21 10 <png>" sobre esa mitad frontal (líneas "0 !:").
#   76382.dat "Minifig Torso with Arms and Hands"; 73200b-f1.dat "Minifig Hips and Legs (Standing)".
#   Pelo: piezas "Minifig Hair ..." con el mismo origen que la cabeza (cajas medidas en la librería).
# Especificación !TEXMAP: ldraw.org/texmap-spec.html (la textura se busca en textures/<png> y luego <png>).
LDRAW_URL = "https://library.ldraw.org/library/updates/complete.zip"
LDRAW_ZIP = Path(os.environ.get("FLUX_CARA_LDRAW_ZIP", "/tmp/ldraw/complete.zip"))   # fuera de /kaggle/working
LDRAW_USER_AGENT = "Mozilla/5.0 (compatible; flux-cara-ldraw/1.0)"   # el UA por defecto de Python recibió 403
LDRAW_BUSCAR_EN = ["/kaggle/input"]  # plan B: un complete.zip subido como Input se usa sin descargar
CABEZA = "3626c.dat"
TEXMAP_PLANAR = "-13 0 10 13 0 10 -13 21 10"
TORSO = "76382.dat"                  # se copian sus líneas para dar a las manos su propio color
PIERNAS = "73200b-f1.dat"
POS_CABEZA = (0, -24, 0)             # cabeza y -4..24 apoyada sobre el torso (y -12..32)
POS_PIERNAS = (0, 32, 0)             # caderas bajo el torso
COLOR_CABEZA = "Yellow"              # nombres de LDConfig.ldr (el código se lee de ese archivo)
COLOR_MANOS = "Yellow"
COLOR_TORSO = "Red"
COLOR_PIERNAS = "Blue"
# Catálogo de cabello: estilo -> descripciones a buscar en la librería (se usa la primera que exista)
CATALOGO_CABELLO = {
    "corto": ["Minifig Hair Short Tousled", "Minifig Hair Male"],
    "largo": ["Minifig Hair Female Long Straight", "Minifig Hair Long Straight"],
    "rizado": ["Minifig Hair Bubble Style (Afro)", "Minifig Hair Coiled"],
    "coleta": ["Minifig Hair Ponytail"],
    "bob": ["Minifig Hair Short Bob Cut"],
    "calvo": [],
}
# PairLegoFace no trae cabello: estilo por defecto según 'femenino' (supuesto) salvo que lo pases a mano
CABELLO_POR_DEFECTO = {True: "largo", False: "corto", None: "corto"}
CABELLO_MANUAL = {}                  # {"3626cp7a_04.jpg": "rizado"}
COLOR_PELO_LDRAW = {                 # color de cejas (texto) -> nombre en LDConfig.ldr
    "black": "Black", "dark brown": "Dark_Brown", "brown": "Reddish_Brown", "reddish brown": "Reddish_Brown",
    "light brown": "Medium_Nougat", "tan": "Tan", "blonde": "Tan", "yellow": "Bright_Light_Yellow",
    "white": "White", "gray": "Light_Bluish_Grey", "dark gray": "Dark_Bluish_Grey", "red": "Dark_Orange",
    "orange": "Dark_Orange", "ginger": "Dark_Orange", "auburn": "Reddish_Brown",
}
COLOR_PELO_POR_DEFECTO = "Black"
LADO_PREVIA = 600                    # vista previa frontal aproximada (no reemplaza el render)

## Módulos

In [ ]:
%%writefile /kaggle/working/flux_cara/src/registro.py
"""Un solo reporte y un solo zip para todo el notebook.

    registro.iniciar()            # al comienzo: todo lo impreso va también a flux_cara/reporte.txt
    registro.seccion("1.1 ...")   # títulos
    registro.mostrar(df)          # display() que además deja la tabla en el reporte
    registro.empaquetar()         # flux_cara/entregar.zip con todo lo que hay hasta ahora (se llama al final de cada etapa)

Si una celda falla, el error se escribe en el reporte y el zip se arma igual (para mandarlo tal cual).
"""
import sys
import traceback
import zipfile
from datetime import datetime

import config as C

EXTENSIONES = (".txt", ".json", ".csv", ".jpg", ".png", ".ldr", ".dat", ".md")
EXCLUIR = ("src", "crudas")            # carpetas que no van al zip (código y generaciones crudas de 512 px)


class _Tee:
    def __init__(self, original, archivo):
        self.original, self.archivo = original, archivo

    def write(self, texto):
        self.original.write(texto)
        try:
            self.archivo.write(texto)
            self.archivo.flush()
        except ValueError:                        # archivo cerrado
            pass
        return len(texto)

    def flush(self):
        self.original.flush()

    def __getattr__(self, nombre):                # isatty, encoding, etc.
        return getattr(self.original, nombre)


def detener():
    if isinstance(sys.stdout, _Tee):
        sys.stdout.archivo.close()
        sys.stdout = sys.stdout.original


def _al_fallar(shell, etype, evalue, tb, tb_offset=None):
    texto = "".join(traceback.format_exception(etype, evalue, tb))
    if isinstance(sys.stdout, _Tee):
        sys.stdout.archivo.write("\n" + "!" * 80 + "\n❌ ERROR en la celda:\n" + texto + "!" * 80 + "\n")
        sys.stdout.archivo.flush()
    try:
        empaquetar()
    except Exception as e:                        # nunca tapar el error original
        sys.__stdout__.write(f"(no se pudo armar el zip: {e})\n")
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


def iniciar(nombre="reporte"):
    """Empieza el reporte (flux_cara/reporte.txt) y engancha los errores de celda."""
    detener()
    C.SALIDA.mkdir(parents=True, exist_ok=True)
    f = open(C.SALIDA / f"{nombre}.txt", "w", encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, f)
    print(f"# Reporte · {datetime.now():%Y-%m-%d %H:%M}")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None:
            ip.set_custom_exc((Exception,), _al_fallar)
    except ImportError:
        pass


def seccion(titulo):
    print("\n" + "=" * 80 + f"\n{titulo}\n" + "=" * 80)


def mostrar(obj, filas=12):
    """Muestra en el notebook y deja una versión en texto en el reporte."""
    try:
        from IPython.display import display
        display(obj)
    except ImportError:
        pass
    if hasattr(obj, "to_string"):
        texto = obj.head(filas).to_string(max_colwidth=70)
    else:
        texto = f"[{type(obj).__name__} {getattr(obj, 'size', '')}]"
    if isinstance(sys.stdout, _Tee):
        sys.stdout.archivo.write(texto + "\n")
        sys.stdout.archivo.flush()


def empaquetar(nombre="entregar", max_mb=20):
    """flux_cara/entregar.zip con reporte + imágenes + tablas + archivos LDraw (sin src/ ni crudas/)."""
    destino = C.SALIDA / f"{nombre}.zip"
    incluidos, omitidos = [], []
    with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
        for r in sorted(C.SALIDA.rglob("*")):
            rel = r.relative_to(C.SALIDA)
            if not r.is_file() or r.suffix.lower() not in EXTENSIONES or r.name == "reporte.txt":
                continue
            if any(p in EXCLUIR for p in rel.parts[:-1]):
                continue
            if r.stat().st_size > max_mb * 2**20:
                omitidos.append(str(rel))
                continue
            z.write(r, rel.as_posix())
            incluidos.append(rel.as_posix())
        print(f"\n📦 {destino} · {len(incluidos)} archivos + reporte.txt"
              + (f" · omitidos por tamaño: {omitidos}" if omitidos else ""))
        print("   → Descárgalo desde el panel Output (flux_cara/entregar.zip) y mándamelo.")
        if isinstance(sys.stdout, _Tee):
            sys.stdout.archivo.flush()
        if (C.SALIDA / "reporte.txt").exists():
            z.write(C.SALIDA / "reporte.txt", "reporte.txt")   # al final, para que incluya todo lo impreso
    return destino

In [ ]:
%%writefile /kaggle/working/flux_cara/src/dataset.py
"""Etapa 0: explorar PairLegoFace leyendo los archivos reales, sin suponer su estructura.

Flujo (cada paso imprime lo que encontró):
    raiz   = encontrar_dataset()
    inv    = inventario(raiz)                  # todos los archivos + tamaño/modo de las imágenes
    tablas = leer_tablas(raiz)                 # CSV/JSON con sus columnas
    fotos  = tabla_fotos(inv, tablas)          # una fila por foto: partición, estampa, rasgos normalizados
    est    = medir_estampas(fotos, inv)        # tamaño y márgenes de las estampas reales
    sel    = elegir_variadas(fotos)            # 10 fotos de prueba con variedad
"""
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps

import config as C

EXT_IMG = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
EXT_TABLA = (".csv", ".tsv", ".json", ".jsonl", ".parquet")
RASGOS = ("lentes", "barba", "bigote", "expresion", "cabello", "color_pelo", "color_cejas")


def carpeta_salida():
    d = C.SALIDA / "datos"
    d.mkdir(parents=True, exist_ok=True)
    return d


# ===================================================================== dónde está el dataset
def encontrar_dataset(raiz=None):
    """Carpeta raíz de PairLegoFace bajo /kaggle/input (o C.RUTA_DATASET si está fijada)."""
    if C.RUTA_DATASET:
        p = Path(C.RUTA_DATASET)
        assert p.exists(), f"RUTA_DATASET no existe: {p}"
        return p
    raiz = Path(raiz or C.ENTRADA)
    assert raiz.exists(), f"No existe {raiz}: ¿agregaste PairLegoFace como Input del notebook?"
    hallados = []
    for actual, dirs, _ in os.walk(raiz):
        nivel = len(Path(actual).relative_to(raiz).parts)
        if nivel >= 4:
            dirs[:] = []
            continue
        for d in dirs:
            if re.search(C.PATRON_NOMBRE_DATASET, d, re.I):
                hallados.append(Path(actual) / d)
    if hallados:
        p = min(hallados, key=lambda h: len(h.parts))           # la menos profunda
        print(f"Dataset: {p}" + (f"  (también: {[str(h) for h in hallados if h != p][:3]})" if len(hallados) > 1 else ""))
        return p
    hijos = [d for d in raiz.iterdir() if d.is_dir()]
    p = hijos[0] if len(hijos) == 1 else raiz
    print(f"⚠ Ninguna carpeta se llama como '{C.PATRON_NOMBRE_DATASET}'. Uso {p}. Fija C.RUTA_DATASET si no es esa.")
    return p


# ===================================================================== inventario de archivos
def inventario(raiz):
    """DataFrame con todos los archivos: ruta, carpeta, ext, bytes y, para imágenes, ancho/alto/modo."""
    filas, leidas = [], Counter()
    for actual, dirs, archivos in os.walk(raiz):
        dirs.sort()
        rel = str(Path(actual).relative_to(raiz)).replace("\\", "/")
        for a in sorted(archivos):
            r = Path(actual) / a
            ext = r.suffix.lower()
            f = dict(ruta=str(r), rel=f"{rel}/{a}", nombre=a, carpeta=rel, ext=ext, bytes=r.stat().st_size,
                     ancho=np.nan, alto=np.nan, modo=None)
            if ext in EXT_IMG and (C.MUESTRA_TAMANOS is None or leidas[rel] < C.MUESTRA_TAMANOS):
                try:
                    with Image.open(r) as im:                    # solo lee la cabecera
                        f["ancho"], f["alto"], f["modo"] = im.width, im.height, im.mode
                        if im.mode == "P" and "transparency" in im.info:
                            f["modo"] = "P+alfa"
                    leidas[rel] += 1
                except Exception as e:                           # noqa: BLE001
                    f["modo"] = f"ilegible: {type(e).__name__}"
            filas.append(f)
    inv = pd.DataFrame(filas)
    print(f"{len(inv)} archivos · {inv['ext'].isin(EXT_IMG).sum()} imágenes · "
          f"{inv['ext'].isin(EXT_TABLA).sum()} tablas · {inv['carpeta'].nunique()} carpetas")
    return inv


def _tamanos(g):
    t = g.dropna(subset=["ancho"])
    if t.empty:
        return ""
    c = Counter(zip(t["ancho"].astype(int), t["alto"].astype(int), t["modo"]))
    return ", ".join(f"{w}x{h} {m} ({n / len(t):.0%})" for (w, h, m), n in c.most_common(3))


def arbol(inv):
    """Resumen por carpeta: cuántos archivos de cada tipo y tamaños de imagen más comunes."""
    lineas = []
    for carpeta, g in inv.groupby("carpeta", sort=True):
        exts = ", ".join(f"{n} {e}" for e, n in g["ext"].value_counts().items())
        t = _tamanos(g)
        lineas.append(f"{carpeta}/  — {exts}" + (f"  · {t}" if t else ""))
    texto = "\n".join(lineas[:C.LINEAS_ARBOL])
    if len(lineas) > C.LINEAS_ARBOL:
        texto += f"\n  … y {len(lineas) - C.LINEAS_ARBOL} carpetas más (sube C.LINEAS_ARBOL para verlas)"
    return texto


# ===================================================================== tablas
def _leer(r):
    ext = Path(r).suffix.lower()
    if ext == ".csv":
        return pd.read_csv(r)
    if ext == ".tsv":
        return pd.read_csv(r, sep="\t")
    if ext == ".jsonl":
        return pd.read_json(r, lines=True)
    if ext == ".parquet":
        return pd.read_parquet(r)
    with open(r, encoding="utf-8") as f:
        d = json.load(f)
    if isinstance(d, list):
        return pd.json_normalize(d)
    if isinstance(d, dict):
        lista = next((v for v in d.values() if isinstance(v, list) and v and isinstance(v[0], dict)), None)
        if lista is not None:
            return pd.json_normalize(lista)
        if all(isinstance(v, dict) for v in d.values()):             # {nombre: {rasgos}}
            return pd.DataFrame.from_dict(d, orient="index").rename_axis("archivo").reset_index()
    raise ValueError("JSON sin forma tabular")


def detectar_columnas(df):
    """{rol: columna} según C.PATRONES_COLUMNA (cada columna va a un solo rol)."""
    usadas, roles = set(), {}
    for rol, patron in C.PATRONES_COLUMNA.items():
        for c in df.columns:
            if c not in usadas and re.search(patron, str(c), re.I):
                roles[rol] = c
                usadas.add(c)
                break
    return roles


def leer_tablas(raiz):
    """Lee cada CSV/JSON del dataset y detecta qué columna es qué."""
    tablas = []
    for r in sorted(Path(raiz).rglob("*")):
        if r.suffix.lower() not in EXT_TABLA or not r.is_file():
            continue
        try:
            df = _leer(r)
        except Exception as e:                                       # noqa: BLE001
            print(f"  {r.relative_to(raiz)}: se omite ({e})")
            continue
        roles = detectar_columnas(df)
        puntos = 3 * ("foto" in roles) + 2 * ("particion" in roles) + ("estampa" in roles) \
            + sum(k in roles for k in ("lentes", "barba", "bigote", "vello", "expresion", "cabello")) \
            + 0.5 * (r.stem.lower() == "pares")
        tablas.append(dict(ruta=r, df=df, roles=roles, puntos=puntos))
        print(f"  {r.relative_to(raiz)}: {len(df)} filas × {len(df.columns)} columnas → {list(df.columns)}")
        print(f"      roles detectados: {roles}")
    if not tablas:
        print("  (sin tablas: la partición y los rasgos saldrán de las carpetas y de C.RASGOS_MANUALES)")
    return sorted(tablas, key=lambda t: -t["puntos"])


# ===================================================================== resolver rutas
class Resolutor:
    """Ruta escrita en una tabla -> archivo real (tal cual, relativa a la tabla o por nombre de archivo)."""

    def __init__(self, rutas):
        self.por_nombre = {}
        for r in rutas:
            self.por_nombre.setdefault(Path(r).name, []).append(Path(r))
        self.ambiguas = set()

    def __call__(self, ruta, base):
        if ruta is None or (isinstance(ruta, float) and np.isnan(ruta)):
            return None
        p = Path(str(ruta).replace("\\", "/"))
        for c in (p, Path(base) / p):
            if c.is_file():
                return c
        hits = self.por_nombre.get(p.name, [])
        if not hits:
            return None
        if len(hits) > 1:
            self.ambiguas.add(p.name)
        # gana la que comparte más carpetas con la ruta escrita; si empatan, la de nombre de carpeta "mayor" (g2 > g1)
        return max(sorted(hits, reverse=True), key=lambda h: sum(x in h.parts for x in p.parts[:-1]))


def imagenes_entrada(raiz=None):
    """Todas las imágenes bajo /kaggle/input (todos los Inputs del notebook), solo rutas."""
    raiz = Path(raiz or C.ENTRADA)
    out = []
    for actual, dirs, archivos in os.walk(raiz):
        out += [os.path.join(actual, a) for a in archivos if a.lower().endswith(EXT_IMG)]
    return out


def describir_tabla(t, muestras=5):
    """Columnas sin rol asignado (valores más comunes) y ejemplos del texto libre."""
    df, roles = t["df"], t["roles"]
    print(f"{t['ruta'].name}: {len(df)} filas · roles {roles}")
    for c in df.columns:
        if c in roles.values() and c not in (roles.get("femenino"), roles.get("edad")):
            continue
        vc = df[c].astype(str).value_counts()
        if len(vc) <= 15:
            print(f"  {c:<15} {vc.to_dict()}")
        else:
            print(f"  {c:<15} {len(vc)} valores distintos, p. ej. {vc.index[:3].tolist()}")
    if "texto" in roles:
        col = roles["texto"]
        print(f"\nEjemplos de '{col}':")
        for x in df[col].dropna().sample(min(muestras, int(df[col].notna().sum())), random_state=0):
            print("  -", str(x)[:400])


def color_cejas(clausula):
    """Color del trozo de la cláusula que menciona las cejas ('white thick eyebrows' -> 'white')."""
    for trozo in re.split(r",| and | with ", clausula):
        if "eyebrow" in trozo or "brow" in trozo:
            for c in C.COLORES_CEJAS:                         # más largo primero
                if re.search(r"\b" + re.escape(c) + r"\b", trozo):
                    return C.MAPEO_COLOR_CEJAS.get(c, c)
    return None


def rasgos_de_texto(texto):
    """Expresión, cabello, color de pelo y de cejas leídos de un texto libre (el prompt con que se generó la foto).
    Si el texto trae la cláusula de rasgos de la cara LEGO (config.PATRON_CLAUSULA_RASGOS), expresión y cejas
    se leen solo de ella."""
    r = dict(expresion=None, cabello=None, color_pelo=None, color_cejas=None, rasgos_lego=None)
    if texto is None or (isinstance(texto, float) and np.isnan(texto)):
        return r
    s = str(texto).lower()
    m = re.search(C.PATRON_CLAUSULA_RASGOS, s)
    clausula = m.group(1).strip() if m else s
    if m:
        r["rasgos_lego"] = clausula
        r["color_cejas"] = color_cejas(clausula)
    r["expresion"] = next((c for c, p in C.PATRONES_EXPRESION_TEXTO if re.search(p, clausula)), None)
    r["cabello"] = next((c for c, p in C.PATRONES_CABELLO_TEXTO if re.search(p, s)), None)
    m = re.search(C.PATRON_COLOR_PELO_TEXTO, s)
    r["color_pelo"] = m.group(1) if m else None
    return r


# ===================================================================== normalizar rasgos
def a_bool(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    s = str(v).strip().lower()
    if s in C.VALORES_SI:
        return True
    if s in C.VALORES_NO or s == "":
        return False
    return None


def norm_expresion(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip().lower()
    if s in C.MAPEO_EXPRESION:
        return C.MAPEO_EXPRESION[s]
    if s in ("sonrisa", "serio", "sorpresa", "neutral"):
        return s
    for cat, patron in C.PATRONES_EXPRESION:
        if re.search(patron, s):
            return cat
    return None


def _texto(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip()
    return s or None


def norm_fila(fila, roles):
    """Fila de la tabla -> dict con lentes, barba, bigote (bool/None), expresion, cabello, color_pelo."""
    g = lambda rol: fila.get(roles[rol]) if rol in roles else None
    r = dict(lentes=None, barba=None, bigote=None, expresion=None, cabello=None, color_pelo=None,
             color_cejas=None, rasgos_lego=None)
    v = g("lentes")
    if v is not None:
        b = a_bool(v)
        r["lentes"] = b if b is not None else not re.search(r"^no|sin|none", str(v).lower())
    for k in ("barba", "bigote"):
        if k in roles:
            r[k] = a_bool(g(k))
    if "vello" in roles:                                     # una sola columna tipo "barba_bigote", "candado", "no"
        s = str(g("vello")).lower()
        if r["barba"] is None:
            r["barba"] = bool(re.search(r"barba|beard|candado|goatee|chiva", s))
        if r["bigote"] is None:
            r["bigote"] = bool(re.search(r"bigote|mous|mustach|candado|goatee", s))
    r["expresion"] = norm_expresion(g("expresion"))
    r["cabello"] = _texto(g("cabello"))
    r["color_pelo"] = _texto(g("color_pelo"))
    r["femenino"] = a_bool(g("femenino")) if "femenino" in roles else None
    r["edad"] = _texto(g("edad"))
    r["fuente_texto"] = []
    if "texto" in roles:                                     # completa lo que la tabla no trae
        for k, v in rasgos_de_texto(g("texto")).items():
            if r[k] is None and v is not None:
                r[k] = v
                if k != "rasgos_lego":
                    r["fuente_texto"].append(k)
    return r


def _manual(nombre, ident):
    m = C.RASGOS_MANUALES
    return m.get(nombre) or (m.get(str(ident)) if ident is not None else None) or {}


def _particion(texto):
    partes = re.split(r"[/\\_\-. ]+", str(texto).lower())
    for nombre, palabras in C.PARTICIONES.items():
        if any(p == w or p.startswith(w) for p in partes for w in palabras):
            return nombre
    return None


def es_estampa(rel, modo, ancho, alto):
    """rel = ruta relativa a la raíz del dataset (así 'PairLegoFace' no cuenta como 'lego')."""
    if re.search(C.PALABRAS_ESTAMPA, str(rel), re.I):
        return True
    alfa = modo in ("RGBA", "LA", "PA", "P+alfa")
    return alfa and pd.notna(ancho) and max(ancho, alto) <= C.LADO_MAX_ESTAMPA


# ===================================================================== tabla de fotos
def tabla_fotos(inv, tablas):
    """Una fila por foto con: ruta, nombre, id, particion, estampa, rasgos normalizados y fuente de los rasgos."""
    res = Resolutor(inv.loc[inv["ext"].isin(EXT_IMG), "ruta"])
    t = tablas[0] if tablas and "foto" in tablas[0]["roles"] else None
    if t is not None:
        df, roles, base = t["df"].copy(), t["roles"], t["ruta"].parent
        print(f"Tabla principal: {t['ruta'].name} · roles {roles}")
        df["_ruta"] = [res(r, base) for r in df[roles["foto"]]]
        res_est = res
        if "estampa" in roles:
            if C.BUSCAR_ESTAMPAS_EN_TODO_INPUT:
                res_est = Resolutor(imagenes_entrada())
            unicas = df[roles["estampa"]].dropna().astype(str).unique()
            halladas = sum(res_est(r, base) is not None for r in unicas)
            print(f"  estampas en '{roles['estampa']}': {len(unicas)} distintas, {halladas} encontradas en disco. "
                  f"Ej.: {list(unicas[:3])}")
            if halladas == 0:
                otros = sorted(d.name for d in C.ENTRADA.iterdir() if d.is_dir()) if C.ENTRADA.exists() else []
                print(f"  ⚠ Ninguna estampa está en {C.ENTRADA}. Inputs presentes: {otros}. "
                      "Agrega como Input el dataset que tiene las caras LEGO (si no, se usa el margen por defecto).")
        n0 = len(df)
        if "ok" in roles:                                     # p. ej. columna "pasa": solo las aprobadas
            ok = df[roles["ok"]].map(a_bool)
            df = df[ok.fillna(False).astype(bool)]
            print(f"  columna '{roles['ok']}': quedan {len(df)} de {n0} filas aprobadas")
        if "id" in roles and df[roles["id"]].duplicated().any():
            antes = len(df)
            orden = roles.get("version", roles["foto"])
            df = df.sort_values(orden, ascending=False).drop_duplicates(roles["id"])
            print(f"  {antes - len(df)} filas con id repetido (p. ej. g1 y g2): se queda la de versión mayor")
        filas = []
        for _, f in df.iterrows():
            ruta = f["_ruta"]
            nombre = Path(str(f[roles["foto"]])).name
            ident = f[roles["id"]] if "id" in roles else None
            rasgos = norm_fila(f, roles)
            man = _manual(nombre, ident)
            est = res_est(f[roles["estampa"]], base) if "estampa" in roles else None
            texto = rasgos.pop("fuente_texto")
            filas.append(dict(ruta=str(ruta) if ruta else None, nombre=nombre, id=ident,
                              particion=_particion(f[roles["particion"]]) if "particion" in roles else _particion(ruta),
                              particion_original=f[roles["particion"]] if "particion" in roles else None,
                              estampa=str(est) if est else None,
                              pieza=f["pieza"] if "pieza" in f else (Path(str(est)).stem if est else None),
                              **{**rasgos, **{k: v for k, v in man.items() if k in RASGOS + ("femenino", "edad")}},
                              fuente_rasgos="tabla" + ("+texto(" + ",".join(texto) + ")" if texto else "")
                              + ("+manual" if man else "")))
        fotos = pd.DataFrame(filas)
        faltan = fotos["ruta"].isna().sum()
        if faltan:
            print(f"  ⚠ {faltan} fotos de la tabla no se encontraron en disco (se ignoran). "
                  f"Ej.: {fotos.loc[fotos['ruta'].isna(), 'nombre'].head(3).tolist()}")
        if res.ambiguas:
            print(f"  ⚠ {len(res.ambiguas)} nombres de archivo están en más de una carpeta; se eligió por ruta. "
                  f"Ej.: {sorted(res.ambiguas)[:3]}")
        fotos = fotos.dropna(subset=["ruta"]).drop_duplicates("ruta").reset_index(drop=True)
    else:
        print("Sin tabla con columna de foto: las fotos salen de las carpetas y los rasgos de C.RASGOS_MANUALES")
        img = inv[inv["ext"].isin(EXT_IMG)]
        img = img[[not es_estampa(r, m, w, h) for r, m, w, h in zip(img["rel"], img["modo"], img["ancho"], img["alto"])]]
        filas = []
        for r, nombre, carpeta in zip(img["ruta"], img["nombre"], img["carpeta"]):
            man = _manual(nombre, Path(nombre).stem)
            base = dict.fromkeys(RASGOS + ("femenino", "edad"))
            base.update({k: v for k, v in man.items() if k in RASGOS + ("femenino", "edad")})
            filas.append(dict(ruta=r, nombre=nombre, id=Path(nombre).stem, particion=_particion(carpeta),
                              particion_original=carpeta, estampa=None, pieza=None, **base,
                              fuente_rasgos="manual" if man else "ninguna"))
        fotos = pd.DataFrame(filas)
    resumen_fotos(fotos)
    return fotos


def resumen_fotos(fotos):
    print(f"\n{len(fotos)} fotos encontradas")
    print("Particiones:", fotos["particion"].fillna("(sin partición)").value_counts().to_dict())
    if fotos["particion_original"].notna().any():
        print("  valores originales:", fotos["particion_original"].astype(str).value_counts().head(8).to_dict())
    print("Rasgos (conteo por valor; None = sin dato):")
    for k in RASGOS + ("femenino", "edad"):
        vc = fotos[k].astype(object).where(fotos[k].notna(), None).map(repr).value_counts()
        print(f"  {k:<11} {vc.head(8).to_dict()}")
    print("Fuente de los rasgos:", fotos["fuente_rasgos"].value_counts().to_dict())
    if "rasgos_lego" in fotos and fotos["rasgos_lego"].notna().any():
        trozos = Counter(t.strip() for c in fotos["rasgos_lego"].dropna() for t in re.split(r",| and ", c)
                         if t.strip() and "brow" not in t)
        print("Frases más comunes de la cara LEGO (sin cejas):", dict(trozos.most_common(30)))
        sin = fotos[fotos["expresion"].isna() & fotos["rasgos_lego"].notna()]
        if len(sin):
            print(f"  {len(sin)} fotos sin expresión reconocida. Ej. de su cláusula (ajusta config.PATRONES_EXPRESION_TEXTO):")
            for c in sin["rasgos_lego"].drop_duplicates().head(10):
                print("   ·", c)


# ===================================================================== estampas reales y márgenes
def marco(h, w, frac):
    b = max(1, int(round(min(h, w) * frac)))
    m = np.zeros((h, w), bool)
    m[:b] = m[-b:] = True
    m[:, :b] = m[:, -b:] = True
    return m


def mascara_estampa(ruta):
    """Estampa -> (máscara bool del trazo, cómo se obtuvo). Usa el alfa; si no hay, distancia Lab al fondo."""
    im = ImageOps.exif_transpose(Image.open(ruta))
    if im.mode in ("RGBA", "LA", "PA") or "transparency" in im.info:
        a = np.asarray(im.convert("RGBA"))[..., 3]
        if a.min() < 128:
            return a > 127, "alfa"
    from skimage.color import rgb2lab
    lab = rgb2lab(np.asarray(im.convert("RGB")))
    borde = marco(*lab.shape[:2], C.BORDE_FONDO)
    fondo = np.median(lab[borde], axis=0)
    return np.linalg.norm(lab - fondo, axis=-1) > C.UMBRAL_LAB_ESTAMPA, "fondo_lab"


def rutas_estampas(fotos, inv):
    """Estampas únicas: las que referencia la tabla; si no hay, las imágenes que parecen estampas."""
    if fotos["estampa"].notna().any():
        return sorted(fotos["estampa"].dropna().unique()), "tabla"
    img = inv[inv["ext"].isin(EXT_IMG)]
    sel = [r for r, rel, m, w, h in zip(img["ruta"], img["rel"], img["modo"], img["ancho"], img["alto"])
           if es_estampa(rel, m, w, h)]
    return sorted(set(sel)), "carpetas"


def medir_estampas(fotos, inv):
    """Tamaño y márgenes (px y fracción) de cada estampa real. Guarda etapa0/margenes.json."""
    rutas, origen = rutas_estampas(fotos, inv)
    filas = []
    for r in rutas:
        try:
            m, como = mascara_estampa(r)
        except Exception as e:                                      # noqa: BLE001
            filas.append(dict(ruta=r, error=f"{type(e).__name__}: {e}"))
            continue
        h, w = m.shape
        ys, xs = np.nonzero(m)
        f = dict(ruta=r, ancho=w, alto=h, mascara=como, cobertura=m.mean())
        if ys.size:
            px = dict(arriba=ys.min(), abajo=h - 1 - ys.max(), izquierda=xs.min(), derecha=w - 1 - xs.max())
            f.update({f"{k}_px": v for k, v in px.items()})
            f.update({k: px[k] / (h if k in ("arriba", "abajo") else w) for k in px})
        filas.append(f)
    est = pd.DataFrame(filas) if filas else pd.DataFrame(columns=["ruta"])
    lados = ("arriba", "abajo", "izquierda", "derecha")
    info = dict(n=len(rutas), origen=origen)
    if len(est) and "ancho" in est and est["ancho"].notna().any():
        validas = est.dropna(subset=["arriba"]) if "arriba" in est else est.iloc[0:0]
        cuenta = Counter(zip(est["ancho"].dropna().astype(int), est["alto"].dropna().astype(int)))
        (w, h), n = cuenta.most_common(1)[0]
        info.update(tamano_dominante=[int(w), int(h)], fraccion_tamano_dominante=n / len(est),
                    tamanos=[[int(a), int(b), int(c)] for (a, b), c in cuenta.most_common(5)],
                    n_vacias=int(len(est) - len(validas)),
                    n_error=int(est["error"].notna().sum()) if "error" in est else 0,
                    mascara=est["mascara"].value_counts().to_dict(),
                    cobertura_media=float(est["cobertura"].mean()))
        if len(validas):
            info["fraccion"] = {k: float(validas[k].mean()) for k in lados}
            info["fraccion_mediana"] = {k: float(validas[k].median()) for k in lados}
            info["px"] = {k: float(validas[f"{k}_px"].mean()) for k in lados}
    if "fraccion" not in info:
        info["fraccion"] = dict(C.MARGEN_POR_DEFECTO)
        info["margen_por_defecto"] = True
        print(f"⚠ No hay estampas reales medibles: se usará el margen por defecto {C.MARGEN_POR_DEFECTO}")
    info["tamano_final"] = list(C.TAMANO_FINAL or info.get("tamano_dominante") or C.TAMANO_POR_DEFECTO)
    d = carpeta_salida()
    est.to_csv(d / "estampas.csv", index=False)
    (d / "margenes.json").write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{info['n']} estampas reales (desde {origen})")
    if "tamano_dominante" in info:
        w, h = info["tamano_dominante"]
        print(f"  tamaño más común: {w} de ancho × {h} de alto ({info['fraccion_tamano_dominante']:.0%}); "
              f"otros: {info['tamanos'][1:]}")
        print(f"  trazo obtenido por: {info['mascara']} · vacías: {info['n_vacias']} · "
              f"cobertura media del trazo: {info['cobertura_media']:.1%}")
    if not info.get("margen_por_defecto"):
        print("  margen medio  " + " · ".join(f"{k} {info['fraccion'][k]:.1%} ({info['px'][k]:.1f} px)" for k in lados))
        print("  margen mediano " + " · ".join(f"{k} {info['fraccion_mediana'][k]:.1%}" for k in lados))
    print(f"  → tamaño final que usará el post-proceso: {info['tamano_final'][0]}×{info['tamano_final'][1]} (ancho × alto)")
    return est, info


# ===================================================================== elegir 10 fotos variadas
def pares_rasgo(fila):
    return {(k, v) for k, v in ((k, fila.get(k)) for k, _ in C.VARIEDAD) if v is not None and not
            (isinstance(v, float) and np.isnan(v))} & set(C.VARIEDAD)


def elegir_variadas(fotos, n=None, semilla=None):
    """n fotos de la partición de prueba que cubran C.VARIEDAD de forma pareja (y caras LEGO distintas)."""
    n = n or C.N_FOTOS
    semilla = C.SEMILLA if semilla is None else semilla
    prueba = fotos[fotos["particion"] == "prueba"]
    if prueba.empty:
        print("⚠ No hay partición de prueba: elijo entre todas las fotos. Revisa C.PARTICIONES.")
        prueba = fotos
    con_rasgos = prueba[prueba[[k for k in RASGOS[:4]]].notna().any(axis=1)]
    if con_rasgos.empty:
        print("⚠ Ninguna foto de prueba tiene rasgos (ni del dataset ni de C.RASGOS_MANUALES): "
              "elijo al azar. Mira la hoja de contactos y completa C.RASGOS_MANUALES.")
    pool = (con_rasgos if len(con_rasgos) >= n else prueba).reset_index(drop=True)
    orden = list(range(len(pool)))
    random.Random(semilla).shuffle(orden)
    elegidas = [i for i in orden if pool.loc[i, "nombre"] in C.FOTOS_FORZADAS]
    conteo, vistos = Counter(), {c: set() for c in C.NOVEDAD}

    def marcar(i):
        conteo.update(pares_rasgo(pool.loc[i]))
        for c in C.NOVEDAD:
            vistos[c].add(pool.loc[i].get(c))

    for i in elegidas:
        marcar(i)
    while len(elegidas) < min(n, len(pool)):
        def puntaje(i):
            f = pool.loc[i]
            return sum(1 / (1 + conteo[p]) for p in pares_rasgo(f)) \
                + sum(0.5 * (bool(pd.notna(f.get(c))) and f.get(c) not in vistos[c]) for c in C.NOVEDAD)
        mejor = max((i for i in orden if i not in elegidas), key=puntaje)   # empate: el primero del orden barajado
        elegidas.append(mejor)
        marcar(mejor)
    sel = pool.loc[elegidas].reset_index(drop=True)
    sel.insert(0, "idx", range(len(sel)))
    print(f"{len(sel)} fotos elegidas de {len(pool)} candidatas (semilla {semilla}). Cobertura:")
    for p in C.VARIEDAD:
        print(f"  {p[0]}={p[1]!r:<12} {conteo[p]} {'✅' if conteo[p] else '— no hay en prueba'}")
    sel.to_csv(carpeta_salida() / "fotos_elegidas.csv", index=False)
    return sel


# ===================================================================== grillas
def cuadrada(ruta, lado):
    im = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    return ImageOps.fit(im, (lado, lado), Image.LANCZOS)


def texto_rasgos(f):
    partes = []
    if f.get("lentes") in (True,):
        partes.append("lentes")
    if f.get("barba") in (True,):
        partes.append("barba")
    if f.get("bigote") in (True,):
        partes.append("bigote")
    if f.get("femenino") in (True,):
        partes.append("mujer")
    for k in ("expresion", "cabello", "edad", "color_cejas"):
        if f.get(k) is not None and not (isinstance(f.get(k), float) and np.isnan(f.get(k))):
            partes.append(str(f[k]))
    return " ".join(partes) or "(sin rasgos)"


def grilla(fotos, cols=5, lado=200, con_estampa=None):
    """Foto (y su estampa real al lado, si existe) con el índice y los rasgos debajo."""
    if con_estampa is None:
        con_estampa = bool(fotos["estampa"].notna().any()) if "estampa" in fotos else False
    ancho_celda = lado * (2 if con_estampa else 1)
    filas = (len(fotos) + cols - 1) // cols
    hoja = Image.new("RGB", (cols * ancho_celda, filas * (lado + 30)), "white")
    d = ImageDraw.Draw(hoja)
    for k, (_, f) in enumerate(fotos.iterrows()):
        x, y = (k % cols) * ancho_celda, (k // cols) * (lado + 30)
        hoja.paste(cuadrada(f["ruta"], lado), (x, y))
        if con_estampa and f.get("estampa"):
            e = Image.open(f["estampa"]).convert("RGBA")
            e.thumbnail((lado - 10, lado - 10), Image.NEAREST)
            fondo = Image.new("RGBA", (lado, lado), (235, 235, 235, 255))
            fondo.alpha_composite(e, ((lado - e.width) // 2, (lado - e.height) // 2))
            hoja.paste(fondo.convert("RGB"), (x + lado, y))
        d.text((x + 4, y + lado + 2), f"#{f.get('idx', k)} {str(f['nombre'])[:28]}", fill="black")
        d.text((x + 4, y + lado + 15), texto_rasgos(f)[:40], fill=(0, 90, 160))
    return hoja


def hoja_estampas(rutas, n=24, cols=8, escala=2):
    """Muestra de estampas reales sobre damero (para ver alfa y formato)."""
    rutas = list(rutas)[:n]
    if not rutas:
        return None
    ims = [Image.open(r).convert("RGBA") for r in rutas]
    w = max(i.width for i in ims) * escala
    h = max(i.height for i in ims) * escala
    filas = (len(ims) + cols - 1) // cols
    hoja = Image.new("RGBA", (cols * w, filas * h), "white")
    for k, im in enumerate(ims):
        x, y = (k % cols) * w, (k // cols) * h
        cuadros = Image.new("RGBA", (w, h))
        dd = ImageDraw.Draw(cuadros)
        for i in range(0, w, 8):
            for j in range(0, h, 8):
                dd.rectangle([i, j, i + 7, j + 7], fill=(200, 200, 200, 255) if (i + j) // 8 % 2 else (250, 250, 250, 255))
        cuadros.alpha_composite(im.resize((im.width * escala, im.height * escala), Image.NEAREST))
        hoja.paste(cuadros, (x, y))
    return hoja.convert("RGB")


# ===================================================================== informe
def guardar_informe(raiz, inv, tablas, fotos, info_estampas, sel):
    d = carpeta_salida()
    inv.to_csv(d / "inventario.csv", index=False)
    fotos.to_csv(d / "fotos.csv", index=False)
    informe = dict(
        dataset=str(raiz),
        archivos=int(len(inv)),
        imagenes=int(inv["ext"].isin(EXT_IMG).sum()),
        tablas=[dict(archivo=str(t["ruta"].relative_to(raiz)), filas=len(t["df"]),
                     columnas=[str(c) for c in t["df"].columns], roles={k: str(v) for k, v in t["roles"].items()})
                for t in tablas],
        fotos=int(len(fotos)),
        particiones=fotos["particion"].fillna("(sin)").value_counts().to_dict(),
        rasgos_disponibles=[k for k in RASGOS if fotos[k].notna().any()],
        estampas=info_estampas,
        elegidas=sel["nombre"].tolist(),
    )
    (d / "informe.json").write_text(json.dumps(informe, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return informe


def verificar(informe):
    """Autoverificación de la Etapa 0: imprime ✅/❌ y devuelve True si todo está bien."""
    d = carpeta_salida()
    pruebas = [
        ("se encontró el dataset", Path(informe["dataset"]).exists()),
        ("hay fotos", informe["fotos"] > 0),
        ("hay partición de prueba", informe["particiones"].get("prueba", 0) > 0),
        (f"se eligieron {C.N_FOTOS} fotos", len(informe["elegidas"]) == C.N_FOTOS),
        ("hay rasgos (dataset o manuales)", bool(informe["rasgos_disponibles"])),
        ("márgenes medidos en estampas reales", not informe["estampas"].get("margen_por_defecto")),
        ("grilla guardada", (d / "grilla_10.jpg").exists()),
        ("margenes.json guardado", (d / "margenes.json").exists()),
    ]
    for nombre, ok in pruebas:
        print(("✅ " if ok else "❌ ") + nombre)
    return all(ok for _, ok in pruebas)

In [ ]:
%%writefile /kaggle/working/flux_cara/src/flux.py
"""Etapa 1: FLUX.2-klein-4B (sin entrenar) foto -> estampa de cara sobre fondo amarillo.

    pipe = flux.cargar()                       # UNA sola vez; los reload del módulo no lo pierden
    tabla = flux.prompts(ELEGIDAS)             # prompt de cada foto según sus rasgos (config.FRASES)
    res = flux.probar(pipe, tabla, "etapa1")   # genera, guarda PNG y mide el tiempo por imagen
    flux.grilla_pares(tabla, res)              # foto | resultado

Lo verificado en la ficha oficial está anotado en config.py (sección Etapa 1).
"""
import inspect
import sys
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import config as C
from dataset import cuadrada, texto_rasgos

# El pipeline vive fuera del módulo: importlib.reload(flux) no obliga a volver a cargar 16 GB.
_ESTADO = sys.modules.setdefault("_flux_cara_estado", types.ModuleType("_flux_cara_estado"))
if not hasattr(_ESTADO, "pipe"):
    _ESTADO.pipe = None


def carpeta(etapa):
    d = C.SALIDA / etapa
    d.mkdir(parents=True, exist_ok=True)
    return d


# ===================================================================== carga
def _gpus():
    import torch
    if not torch.cuda.is_available():
        return "sin GPU"
    return " · ".join(f"cuda:{i} {torch.cuda.get_device_name(i)} "
                      f"{torch.cuda.memory_allocated(i) / 2**30:.1f}/{torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GiB"
                      for i in range(torch.cuda.device_count()))


def cargar(forzar=False):
    """Carga el pipeline una sola vez (DiffusionPipeline.from_pretrained con device_map) y lo reutiliza."""
    if _ESTADO.pipe is not None and not forzar:
        print("Pipeline ya cargado: se reutiliza.")
        return _ESTADO.pipe
    import torch
    import diffusers
    from diffusers import DiffusionPipeline
    print("diffusers", diffusers.__version__, "· torch", torch.__version__)
    print("GPUs antes:", _gpus())
    kw = dict(torch_dtype=getattr(torch, C.DTYPE))
    if C.DEVICE_MAP:
        kw["device_map"] = C.DEVICE_MAP
    if C.MAX_MEMORY:
        kw["max_memory"] = C.MAX_MEMORY
    t = time.time()
    pipe = DiffusionPipeline.from_pretrained(C.MODELO_ID, **kw)
    print(f"{type(pipe).__name__} cargado en {time.time() - t:.0f} s · {kw}")
    mapa = getattr(pipe, "hf_device_map", None)
    print("hf_device_map:", mapa)
    if mapa and any(str(v) in ("cpu", "disk") for v in mapa.values()):
        print("⚠ Hay componentes en CPU/disco: va a ser lento. Revisa config.MAX_MEMORY.")
    print("GPUs después:", _gpus())
    try:
        pipe.set_progress_bar_config(disable=True)
    except Exception:
        pass
    _ESTADO.pipe = pipe
    return pipe


def verificar_firma(pipe):
    """Comprueba en la versión instalada que __call__ acepta la imagen de referencia y los parámetros que usamos."""
    params = inspect.signature(pipe.__call__).parameters
    faltan = [p for p in ("prompt", "image", "height", "width", "num_inference_steps", "guidance_scale", "generator")
              if p not in params]
    ok = not faltan
    print(("✅" if ok else "❌") + f" {type(pipe).__name__}.__call__ acepta image= (referencia)"
          + ("" if ok else f" · faltan: {faltan}"))
    if "image" in params:
        print("   anotación de image:", params["image"].annotation)
    return ok


# ===================================================================== prompt
def _vacio(v):
    return v is None or (isinstance(v, float) and np.isnan(v))


def _si(v):
    if _vacio(v):
        return None
    if isinstance(v, str):
        return v.strip().lower() in C.VALORES_SI
    return bool(v)


def frases(fila):
    """Rasgos de una foto -> frases del prompt + qué se rellenó por defecto."""
    defecto = []
    lentes = _si(fila.get("lentes"))
    if lentes is None:
        defecto.append("lentes")
    barba, bigote = _si(fila.get("barba")), _si(fila.get("bigote"))
    if barba is None or bigote is None:
        defecto.append("vello")
    vello = [C.FRASES["barba"]] * bool(barba) + [C.FRASES["bigote"]] * bool(bigote)
    expr = fila.get("expresion")
    if _vacio(expr) or expr not in C.FRASES["expresion"]:
        defecto.append("expresion")
        expr = C.EXPRESION_POR_DEFECTO
    color = fila.get("color_cejas")
    if _vacio(color):
        defecto.append("color")
        color = C.COLOR_CEJAS_POR_DEFECTO
    return dict(expresion=C.FRASES["expresion"][expr], lentes=C.FRASES["lentes"][bool(lentes)],
                vello=C.UNIR_VELLO.join(vello) if vello else C.FRASE_SIN_VELLO, color=color), defecto


def prompts(fotos, plantilla=None):
    """Tabla con el prompt de cada foto (idx, nombre, ruta, rasgos, frases, prompt, por_defecto)."""
    plantilla = plantilla or C.PROMPT_BASE
    filas = []
    for k, (_, f) in enumerate(fotos.iterrows()):
        fr, defecto = frases(f)
        filas.append(dict(idx=f.get("idx", k), nombre=f["nombre"], ruta=f["ruta"], rasgos=texto_rasgos(f),
                          **{c: f.get(c) for c in ("lentes", "barba", "bigote", "femenino", "expresion", "color_cejas")},
                          rasgos_lego=f.get("rasgos_lego"), **{"frase_" + a: b for a, b in fr.items()},
                          prompt=plantilla.format(**fr), por_defecto=",".join(defecto)))
    t = pd.DataFrame(filas)
    n = (t["por_defecto"] != "").sum()
    print(f"{len(t)} prompts armados" + (f" · {n} con algún rasgo por defecto (columna por_defecto)" if n else ""))
    return t


# ===================================================================== generar
def _sincronizar():
    import torch
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            torch.cuda.synchronize(i)


def generar(pipe, ruta_foto, prompt, semilla):
    """Una imagen: foto de referencia (recorte cuadrado) + prompt. Devuelve (PIL RGB, segundos)."""
    import torch
    ref = cuadrada(ruta_foto, C.LADO_REFERENCIA)
    gen = torch.Generator(C.GENERADOR_EN).manual_seed(int(semilla))
    _sincronizar()
    t = time.perf_counter()
    salida = pipe(prompt=prompt, image=ref, height=C.ALTO, width=C.ANCHO, num_inference_steps=C.PASOS,
                  guidance_scale=C.GUIA, generator=gen)
    _sincronizar()
    return salida.images[0].convert("RGB"), time.perf_counter() - t


def probar(pipe, tabla, etapa, etiqueta="base", semilla=None):
    """Genera una imagen por fila de la tabla; guarda etapa/etiqueta/idx_nombre.png. Devuelve la tabla con tiempos."""
    semilla = C.SEMILLA_FLUX if semilla is None else semilla
    d = carpeta(etapa) / "crudas" / etiqueta          # crudas/ no va al zip (pesan); las grillas sí
    d.mkdir(parents=True, exist_ok=True)
    filas = []
    for _, f in tabla.iterrows():
        im, s = generar(pipe, f["ruta"], f["prompt"], semilla)
        destino = d / f"{f['idx']:02d}_{Path(f['nombre']).stem}.png"
        im.save(destino)
        a = np.asarray(im)
        filas.append(dict(idx=f["idx"], nombre=f["nombre"], semilla=semilla, segundos=round(s, 2), salida=str(destino),
                          ancho=im.width, alto=im.height, media_rgb=[round(x) for x in a.reshape(-1, 3).mean(0)]))
        print(f"  #{f['idx']} {f['nombre']}: {s:.1f} s")
    r = pd.DataFrame(filas)
    t = r["segundos"]
    print(f"Tiempo por imagen: primera {t.iloc[0]:.1f} s (incluye calentamiento) · resto media {t.iloc[1:].mean():.1f} s "
          f"(mín {t.iloc[1:].min():.1f}, máx {t.iloc[1:].max():.1f}) · total {t.sum():.0f} s" if len(t) > 1 else
          f"Tiempo: {t.iloc[0]:.1f} s")
    return r


# ===================================================================== grilla
def grilla_pares(tabla, res, cols=5, lado=200):
    """Foto | resultado, con índice, rasgos y tiempo debajo."""
    res = res.set_index("idx")
    ancho = lado * 2 + 8
    filas = (len(tabla) + cols - 1) // cols
    hoja = Image.new("RGB", (cols * ancho, filas * (lado + 44)), "white")
    d = ImageDraw.Draw(hoja)
    for k, (_, f) in enumerate(tabla.iterrows()):
        x, y = (k % cols) * ancho, (k // cols) * (lado + 44)
        hoja.paste(cuadrada(f["ruta"], lado), (x, y))
        r = res.loc[f["idx"]]
        hoja.paste(cuadrada(r["salida"], lado), (x + lado, y))
        d.text((x + 4, y + lado + 2), f"#{f['idx']} {str(f['nombre'])[:22]}  {r['segundos']:.1f}s", fill="black")
        d.text((x + 4, y + lado + 15), str(f["rasgos"])[:44], fill=(0, 90, 160))
        extra = f"cejas {f['frase_color']}" + (f" · defecto: {f['por_defecto']}" if f["por_defecto"] else "")
        d.text((x + 4, y + lado + 28), extra[:44], fill=(150, 60, 0))
    return hoja


# ===================================================================== verificación
def verificar(tabla, res, firma_ok):
    chequeos = [
        ("pipeline acepta image= (foto de referencia)", firma_ok),
        (f"{len(tabla)} imágenes generadas", len(res) == len(tabla) and len(res) > 0),
        (f"tamaño {C.ANCHO}x{C.ALTO}", bool(((res["ancho"] == C.ANCHO) & (res["alto"] == C.ALTO)).all())),
        ("ninguna imagen negra/vacía (NaN en fp16 da negro)", all(max(m) > 10 for m in res["media_rgb"])),
        ("tiempo medido en todas", bool(res["segundos"].notna().all())),
    ]
    for texto, ok in chequeos:
        print(("✅ " if ok else "❌ ") + texto)
    return all(ok for _, ok in chequeos)

In [ ]:
%%writefile /kaggle/working/flux_cara/src/postproceso.py
"""Etapa 3: imagen de FLUX (fondo amarillo) -> estampa RGBA del tamaño de las estampas reales.

    info = postproceso.analizar(img)                  # Lab, fondo = mediana del borde, trazo, guardrail
    r = postproceso.a_rgba(img, margenes, tamano)     # r["rgba"] (PIL RGBA) o r["descartada"] con r["motivo"]

Pasos (especificación): Lab (skimage.color.rgb2lab) · fondo = mediana del 5 % exterior · trazo = distancia Lab > 25
· apertura 3x3 · se borran componentes < 15 px · encuadre con los márgenes medidos en las estampas reales ·
tamaño final con alfa NEAREST y color BOX · guardrail: trazo 2–50 % del área, >= 60 % del borde es fondo y
sin cabeza dibujada (silueta rellena más grande <= 30 % de la imagen).
"""
import json

import numpy as np
from PIL import Image, ImageDraw
from scipy import ndimage
from skimage.color import rgb2lab

import config as C


def anillo(h, w, frac=None):
    """Máscara del borde exterior (frac del lado menor)."""
    b = max(1, int(round((C.BORDE_FONDO if frac is None else frac) * min(h, w))))
    m = np.zeros((h, w), bool)
    m[:b], m[-b:], m[:, :b], m[:, -b:] = True, True, True, True
    return m


def analizar(img):
    """Separa trazo y fondo. Devuelve dict con trazo (bool HxW), fondo_rgb, cobertura, borde_fondo y descarte."""
    rgb = np.asarray(img.convert("RGB"))
    lab = rgb2lab(rgb / 255.0)
    h, w = rgb.shape[:2]
    borde = anillo(h, w)
    fondo = np.median(lab[borde], axis=0)
    dist = np.linalg.norm(lab - fondo, axis=-1)
    trazo = dist > C.UMBRAL_LAB
    borde_fondo = float((~trazo[borde]).mean())               # antes de limpiar: ¿cuánto del borde es fondo?
    k = np.ones((C.APERTURA, C.APERTURA), bool)
    trazo = ndimage.binary_opening(trazo, structure=k)
    etiq, n = ndimage.label(trazo, structure=np.ones((3, 3)))
    if n:
        tam = np.bincount(etiq.ravel())
        chicos = tam < C.MIN_COMPONENTE
        chicos[0] = False
        trazo[chicos[etiq]] = False
    cobertura = float(trazo.mean())
    silueta = silueta_mayor(trazo)
    fondo_rgb = np.median(rgb[borde], axis=0)
    motivo = []
    if silueta > C.SILUETA_MAX:
        motivo.append(f"cabeza dibujada (silueta {silueta:.0%} > {C.SILUETA_MAX:.0%})")
    if cobertura < C.COBERTURA_MIN:
        motivo.append(f"trazo {cobertura:.1%} < {C.COBERTURA_MIN:.0%}")
    if cobertura > C.COBERTURA_MAX:
        motivo.append(f"trazo {cobertura:.1%} > {C.COBERTURA_MAX:.0%}")
    if borde_fondo < C.BORDE_FONDO_MIN:
        motivo.append(f"borde fondo {borde_fondo:.0%} < {C.BORDE_FONDO_MIN:.0%}")
    return dict(rgb=rgb, lab=lab, trazo=trazo, dist=dist, fondo_rgb=fondo_rgb, cobertura=cobertura,
                borde_fondo=borde_fondo, silueta=silueta, descartada=bool(motivo), motivo="; ".join(motivo))


def silueta_mayor(trazo):
    """Área (fracción de la imagen) de la figura rellena más grande del trazo.

    Una cabeza dibujada encierra la cara (contorno, pelo + barba) o viene rellena de piel: al cerrar huecos
    chicos y rellenar lo encerrado queda una silueta de ~40-50 % de la imagen. Ojos, cejas, boca, lentes o una
    barba en U quedan por debajo de ~20 % (medido en las grillas de la corrida de Kaggle del 2026-11-16)."""
    if not trazo.any():
        return 0.0
    cerrado = ndimage.binary_closing(trazo, structure=np.ones((3, 3)), iterations=C.SILUETA_CIERRE)
    etiq, n = ndimage.label(ndimage.binary_fill_holes(cerrado))
    if not n:
        return 0.0
    tam = np.bincount(etiq.ravel())
    tam[0] = 0
    return float(tam.max() / trazo.size)


def margenes(info_estampas=None, ruta_json=None):
    """Márgenes (fracción) y tamaño final (ancho, alto) desde lo medido en las estampas reales."""
    if info_estampas is None and ruta_json is not None and ruta_json.exists():
        info_estampas = json.loads(ruta_json.read_text(encoding="utf-8"))
    info_estampas = info_estampas or {}
    if info_estampas.get("margen_por_defecto") or not info_estampas.get("fraccion"):
        m, origen = dict(C.MARGEN_POR_DEFECTO), "por defecto (sin estampas reales)"
    else:
        clave = "fraccion_mediana" if C.ESTADISTICO_MARGEN == "mediana" else "fraccion"
        m, origen = dict(info_estampas.get(clave) or info_estampas["fraccion"]), f"{C.ESTADISTICO_MARGEN} de {info_estampas.get('n')} estampas reales"
    tam = tuple(C.TAMANO_FINAL or info_estampas.get("tamano_final") or C.TAMANO_POR_DEFECTO)
    return m, tam, origen


def a_rgba(img, marg, tamano):
    """Imagen -> dict(rgba, descartada, motivo, cobertura, borde_fondo, caja). Encuadre: alto según márgenes
    arriba/abajo, ancho centrado sin deformar y con al menos C.MARGEN_LATERAL_MIN a cada lado."""
    a = analizar(img)
    W, H = tamano
    res = dict(descartada=a["descartada"], motivo=a["motivo"], cobertura=a["cobertura"],
               borde_fondo=a["borde_fondo"], rgba=None, caja=None, trazo=a["trazo"])
    t = a["trazo"]
    if not t.any():
        res.update(descartada=True, motivo=(a["motivo"] + "; " if a["motivo"] else "") + "sin trazo")
        return res
    filas, cols = np.where(t.any(1))[0], np.where(t.any(0))[0]
    y0, y1, x0, x1 = filas[0], filas[-1] + 1, cols[0], cols[-1] + 1
    mascara = t[y0:y1, x0:x1]
    color = a["rgb"][y0:y1, x0:x1].copy()
    # el fondo toma el color del trazo más cercano: así BOX no mezcla amarillo en los bordes del trazo
    _, (iy, ix) = ndimage.distance_transform_edt(~mascara, return_indices=True)
    color = color[iy, ix]
    alto_caja = H * (1 - marg["arriba"] - marg["abajo"])
    ancho_max = W * (1 - 2 * C.MARGEN_LATERAL_MIN)
    esc = min(alto_caja / mascara.shape[0], ancho_max / mascara.shape[1])
    nw, nh = max(1, round(mascara.shape[1] * esc)), max(1, round(mascara.shape[0] * esc))
    col = Image.fromarray(color).resize((nw, nh), Image.BOX)
    alfa = Image.fromarray(mascara.astype(np.uint8) * 255).resize((nw, nh), Image.NEAREST)
    x = (W - nw) // 2
    y = int(round(H * marg["arriba"] + (alto_caja - nh) / 2))
    capa = col.convert("RGBA")
    capa.putalpha(alfa)
    lienzo = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    lienzo.paste(capa, (x, y))
    res.update(rgba=lienzo, caja=(x, y, nw, nh))
    return res


def tablero(w, h, c=8):
    t = Image.new("RGB", (w, h), (230, 230, 230))
    d = ImageDraw.Draw(t)
    for yy in range(0, h, c):
        for xx in range((yy // c % 2) * c, w, 2 * c):
            d.rectangle([xx, yy, xx + c - 1, yy + c - 1], fill=(200, 200, 200))
    return t


def sobre_tablero(rgba, ancho):
    esc = ancho / rgba.width
    im = rgba.resize((ancho, round(rgba.height * esc)), Image.NEAREST)
    fondo = tablero(*im.size).convert("RGBA")
    fondo.alpha_composite(im)
    return fondo.convert("RGB")


def grilla(pares, etiquetas, cols=5, lado=200):
    """pares = [(imagen original, resultado de a_rgba)]: original | máscara | RGBA sobre tablero."""
    ancho = lado * 3 + 8
    filas = (len(pares) + cols - 1) // cols
    hoja = Image.new("RGB", (cols * ancho, filas * (lado + 32)), "white")
    d = ImageDraw.Draw(hoja)
    for k, ((orig, r), et) in enumerate(zip(pares, etiquetas)):
        x, y = (k % cols) * ancho, (k // cols) * (lado + 32)
        hoja.paste(orig.convert("RGB").resize((lado, lado)), (x, y))
        hoja.paste(Image.fromarray((~r["trazo"]).astype(np.uint8) * 255).resize((lado, lado), Image.NEAREST), (x + lado, y))
        if r["rgba"] is not None:
            p = sobre_tablero(r["rgba"], lado)
            hoja.paste(p, (x + 2 * lado, y + (lado - p.height) // 2))
        estado = ("DESCARTADA: " + r["motivo"]) if r["descartada"] else f"ok · trazo {r['cobertura']:.0%} · borde {r['borde_fondo']:.0%}"
        d.text((x + 4, y + lado + 2), str(et)[:60], fill="black")
        d.text((x + 4, y + lado + 16), estado[:70], fill=(170, 0, 0) if r["descartada"] else (0, 120, 0))
    return hoja

In [ ]:
%%writefile /kaggle/working/flux_cara/src/ajuste.py
"""Etapa 2: juzgar cada imagen (¿plana?, ¿fondo amarillo?, ¿solo rasgos?, ¿respeta lentes y barba?), detectar la
falla más común, probar 3 variantes del prompt y congelar la mejor.

El juez es automático (CLIP zero-shot + reglas de color) y por eso es una ESTIMACIÓN: las grillas de cada variante
quedan para revisarlas a ojo y config.REVISION_MANUAL corrige cualquier veredicto.
"""
import colorsys
import sys
import types

import numpy as np
import pandas as pd
from PIL import Image

import config as C
import postproceso

_ESTADO = sys.modules.setdefault("_flux_cara_estado", types.ModuleType("_flux_cara_estado"))
if not hasattr(_ESTADO, "clip"):
    _ESTADO.clip = None

FALLAS = ("3d", "fondo", "cara_completa")


# ===================================================================== CLIP
def _clip():
    if _ESTADO.clip is None:
        import torch
        from transformers import CLIPModel, CLIPProcessor
        disp = "cuda:0" if torch.cuda.is_available() else "cpu"
        modelo = CLIPModel.from_pretrained(C.MODELO_CLIP).to(disp).eval()
        _ESTADO.clip = (modelo, CLIPProcessor.from_pretrained(C.MODELO_CLIP), disp)
        print(f"CLIP {C.MODELO_CLIP} cargado en {disp}")
    return _ESTADO.clip


def clip_probs(img, textos):
    """Probabilidad (softmax) de cada texto para la imagen."""
    import torch
    modelo, proc, disp = _clip()
    ent = proc(text=list(textos), images=img, return_tensors="pt", padding=True)
    ent = {k: v.to(disp) for k, v in ent.items()}
    with torch.no_grad():
        logits = modelo(**ent).logits_per_image
    return logits.float().softmax(dim=-1).cpu().numpy()[0]


# ===================================================================== reglas
def fondo_amarillo(a):
    r, g, b = (np.asarray(a["fondo_rgb"], float) / 255).tolist()
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    h *= 360
    ok = C.AMARILLO_H[0] <= h <= C.AMARILLO_H[1] and s >= C.AMARILLO_S_MIN and v >= C.AMARILLO_V_MIN \
        and a["borde_fondo"] >= C.BORDE_FONDO_MIN
    return ok, round(h), round(s, 2), round(v, 2)


def fraccion_suave(a):
    """Fracción de píxeles del trazo con degradé suave en L* (sombreado típico de 3D/foto)."""
    gy, gx = np.gradient(a["lab"][..., 0])
    g = np.hypot(gx, gy)
    zona = ndilatar(a["trazo"])
    if not zona.any():
        return 0.0
    lo, hi = C.GRADIENTE_SUAVE
    return float(((g > lo) & (g < hi))[zona].mean())


def ndilatar(m, it=2):
    from scipy import ndimage
    return ndimage.binary_dilation(m, iterations=it)


def _objetivo(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, str):
        return v.strip().lower() in C.VALORES_SI
    return bool(v)


def evaluar(img, fila):
    """Veredicto de una imagen. fila = rasgos de la foto (lentes, barba)."""
    a = postproceso.analizar(img)
    amarillo, h, s, v = fondo_amarillo(a)
    suave = fraccion_suave(a)
    r = dict(amarillo=amarillo, fondo_hsv=f"{h},{s},{v}", borde_fondo=round(a["borde_fondo"], 2),
             cobertura=round(a["cobertura"], 3), suave=round(suave, 3), plana_regla=suave <= C.MAX_FRACCION_SUAVE)
    if C.USAR_CLIP:
        p = clip_probs(img, C.CLIP_TEXTOS["plana"])
        r["plana_clip"] = round(float(p[0]), 2)
        r["plana"] = bool(p.argmax() == 0)
        r["solo_rasgos"] = bool(clip_probs(img, C.CLIP_TEXTOS["solo_rasgos"]).argmax() == 0) \
            and a["cobertura"] <= C.COBERTURA_MAX and a["silueta"] <= C.SILUETA_MAX
        r["lentes_det"] = bool(clip_probs(img, C.CLIP_TEXTOS["lentes"]).argmax() == 0)
        r["barba_det"] = bool(clip_probs(img, C.CLIP_TEXTOS["barba"]).argmax() == 0)
    else:
        r.update(plana=r["plana_regla"], solo_rasgos=a["cobertura"] <= C.COBERTURA_MAX and a["silueta"] <= C.SILUETA_MAX,
                 lentes_det=None, barba_det=None)
    r["silueta"] = round(a["silueta"], 3)
    ok = []
    for k in ("lentes", "barba"):
        obj = _objetivo(fila.get(k))
        det = r[k + "_det"]
        ok.append(True if obj is None or det is None else obj == det)
    r["respeta"] = all(ok)
    return r


def evaluar_lote(tabla, res, variante):
    """tabla = prompts (con rasgos de ELEGIDAS unidos), res = salida de flux.probar."""
    filas = []
    for (_, f), (_, r) in zip(tabla.iterrows(), res.iterrows()):
        ev = evaluar(Image.open(r["salida"]), f)
        ev.update(C.REVISION_MANUAL.get(variante, {}).get(int(f["idx"]), {}))
        filas.append(dict(variante=variante, idx=f["idx"], nombre=f["nombre"], **ev))
    e = pd.DataFrame(filas)
    e["plana_y_amarilla"] = e["plana"] & e["amarillo"]
    return e


def falla_principal(ev):
    cuentas = {"3d": int((~ev["plana"]).sum()), "fondo": int((~ev["amarillo"]).sum()),
               "cara_completa": int((~ev["solo_rasgos"]).sum())}
    falla = max(FALLAS, key=lambda k: cuentas[k])          # empate: el orden de FALLAS
    print(f"Fallas en la Etapa 1 (de {len(ev)}): {cuentas} → la más común: '{falla}'")
    return falla, cuentas


def resumen(ev, prompt):
    return dict(variante=ev["variante"].iloc[0], n=len(ev), planas=int(ev["plana"].sum()),
                fondo_amarillo=int(ev["amarillo"].sum()), planas_y_amarillas=int(ev["plana_y_amarilla"].sum()),
                respetan_lentes_y_barba=int(ev["respeta"].sum()), solo_rasgos=int(ev["solo_rasgos"].sum()),
                cobertura_media=round(float(ev["cobertura"].mean()), 3), prompt=prompt)


COLUMNA_FALLA = {"3d": "planas", "fondo": "fondo_amarillo", "cara_completa": "solo_rasgos"}


def elegir(tabla_prompts, falla=None):
    """Congela la variante que mejor corrige la falla detectada.

    Orden: 1) cumple la meta de planas y amarillas · 2) la columna de la falla (p. ej. solo_rasgos para
    'cara_completa': si no, se congela un prompt que no arregla lo que se vino a arreglar) · 3) respetan lentes
    y barba · 4) planas y amarillas · 5) más trazo (a igual puntaje, la que perdió menos rasgos)."""
    t = tabla_prompts.copy()
    t["_meta"] = t["planas_y_amarillas"] >= C.META_PLANAS_AMARILLAS
    col = COLUMNA_FALLA.get(falla)
    orden = ["_meta"] + ([col] if col else []) + ["respetan_lentes_y_barba", "planas_y_amarillas", "cobertura_media"]
    mejor = t.sort_values(orden, ascending=False, kind="stable").iloc[0].drop("_meta")
    meta = bool(mejor["planas_y_amarillas"] >= C.META_PLANAS_AMARILLAS)
    print(f"Criterio: {' > '.join(o.lstrip('_') for o in orden)}")
    print(f"Congelada: '{mejor['variante']}' · planas y amarillas {mejor['planas_y_amarillas']}/{mejor['n']} "
          f"(meta {C.META_PLANAS_AMARILLAS}) {'✅' if meta else '❌ meta no alcanzada'} · "
          f"solo rasgos {mejor['solo_rasgos']}/{mejor['n']} · "
          f"respetan lentes y barba {mejor['respetan_lentes_y_barba']}/{mejor['n']}")
    return mejor, meta

In [ ]:
%%writefile /kaggle/working/flux_cara/src/cara.py
"""Etapa 4: generar_cara(foto, rasgos, n=5) -> n estampas RGBA.

Cinco semillas (SEMILLA_FLUX + k). Si el guardrail descarta una, se reintenta con semilla + 1000, máximo 2 veces;
si las 3 se descartan, queda la última marcada como descartada.
"""
import time
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw

import config as C
import flux
import postproceso


def generar_cara(foto, rasgos, n=None, plantilla=None, pipe=None, marg=None, tamano=None, carpeta=None,
                 semilla_base=None):
    """foto: ruta. rasgos: dict/Series con lentes, barba, bigote, expresion, color_cejas.
    semilla_base: semilla de la cara k=0 (por defecto SEMILLA_FLUX); la Etapa 6 la cambia para la ronda 2.
    Devuelve (lista de n PIL RGBA, DataFrame con semilla, intentos, descartes y segundos por cara)."""
    n = n or C.N_CARAS
    semilla_base = C.SEMILLA_FLUX if semilla_base is None else semilla_base
    plantilla = plantilla or C.PROMPT_BASE
    pipe = pipe or flux.cargar()
    if marg is None or tamano is None:
        marg, tamano, _ = postproceso.margenes(ruta_json=C.SALIDA / "datos" / "margenes.json")
    fr, defecto = flux.frases(rasgos)
    prompt = plantilla.format(**fr)
    caras, filas = [], []
    t_total = time.perf_counter()
    for k in range(n):
        base = semilla_base + k
        intentos, seg = [], 0.0
        for r in range(C.MAX_REINTENTOS + 1):
            semilla = base + r * C.SALTO_REINTENTO
            img, s = flux.generar(pipe, foto, prompt, semilla)
            seg += s
            res = postproceso.a_rgba(img, marg, tamano)
            intentos.append(f"{semilla}:{'descartada (' + res['motivo'] + ')' if res['descartada'] else 'ok'}")
            if not res["descartada"]:
                break
        rgba = res["rgba"] if res["rgba"] is not None else Image.new("RGBA", tamano, (0, 0, 0, 0))
        caras.append(rgba)
        fila = dict(k=k, semilla_final=semilla, reintentos=len(intentos) - 1, descartada=res["descartada"],
                    motivo=res["motivo"], cobertura=round(res["cobertura"], 3), segundos=round(seg, 2),
                    intentos=" | ".join(intentos), por_defecto=",".join(defecto))
        if carpeta is not None:
            carpeta.mkdir(parents=True, exist_ok=True)
            fila["png"] = str(carpeta / f"cara_{k}_s{semilla}.png")
            rgba.save(fila["png"])
            (carpeta / "crudas").mkdir(exist_ok=True)
            img.save(carpeta / "crudas" / f"cruda_{k}_s{semilla}.png")
        filas.append(fila)
    tabla = pd.DataFrame(filas)
    tabla.attrs["segundos_total"] = time.perf_counter() - t_total
    tabla.attrs["prompt"] = prompt
    return caras, tabla


def sobre_amarillo(rgba, lado=224):
    """Estampa RGBA centrada sobre amarillo liso (cuadrada), para juzgarla con CLIP."""
    W, H = rgba.size
    L = max(W, H) + 40
    fondo = Image.new("RGBA", (L, L), (*C.AMARILLO_COMPUESTO, 255))
    fondo.alpha_composite(rgba.convert("RGBA"), ((L - W) // 2, (L - H) // 2))
    return fondo.convert("RGB").resize((lado, lado), Image.BICUBIC)


def puntuar(caras, tabla, rasgos):
    """Por cada cara: ¿tiene lentes? ¿tiene vello (barba o bigote)? según CLIP, y cuántos coinciden con la foto."""
    import ajuste
    obj = dict(lentes=flux._si(rasgos.get("lentes")))
    barba, bigote = flux._si(rasgos.get("barba")), flux._si(rasgos.get("bigote"))
    obj["vello"] = None if barba is None and bigote is None else bool(barba or bigote)
    filas = []
    for k, (c, (_, f)) in enumerate(zip(caras, tabla.iterrows())):
        fila = dict(k=k, semilla=f["semilla_final"], descartada=bool(f["descartada"]), coinciden=0, de=0)
        img = sobre_amarillo(c)
        for r in ("lentes", "vello"):
            p = float(ajuste.clip_probs(img, C.CLIP_TEXTOS[r])[0])
            fila[f"p_{r}"] = round(p, 2)
            if obj[r] is not None:
                fila["de"] += 1
                fila["coinciden"] += int((p > C.UMBRAL_RASGO_CLIP) == obj[r])
        filas.append(fila)
    return pd.DataFrame(filas)


def mejor(caras, tabla, rasgos=None):
    """Cara para la minifig: entre las no descartadas, la que más rasgos de la foto respeta (lentes y vello facial,
    juzgados con CLIP); a igual puntaje, la de menor k. Sin CLIP o sin rasgos: la primera no descartada.
    Devuelve (cara, índice, tabla de puntajes o None)."""
    ok = tabla.index[~tabla["descartada"]]
    candidatas = list(ok) if len(ok) else list(tabla.index)
    if rasgos is None or not C.USAR_CLIP:
        return caras[int(candidatas[0])], int(candidatas[0]), None
    p = puntuar(caras, tabla, rasgos)
    p["elegible"] = p["k"].isin(candidatas)
    i = int(p[p["elegible"]].sort_values(["coinciden", "k"], ascending=[False, True]).iloc[0]["k"])
    p["elegida"] = p["k"] == i
    return caras[i], i, p.drop(columns="elegible")


def tira(foto, caras, tabla, lado=160):
    """Foto + sus n estampas (sobre tablero), con semilla y estado."""
    hoja = Image.new("RGB", ((len(caras) + 1) * lado, lado + 30), "white")
    hoja.paste(flux.cuadrada(foto, lado), (0, 0))
    d = ImageDraw.Draw(hoja)
    for k, (c, (_, f)) in enumerate(zip(caras, tabla.iterrows()), start=1):
        p = postproceso.sobre_tablero(c, lado - 8)
        hoja.paste(p, (k * lado + 4, (lado - p.height) // 2))
        d.text((k * lado + 4, lado + 2), f"s{f['semilla_final']} · {f['segundos']:.1f}s", fill="black")
        d.text((k * lado + 4, lado + 15), "descartada" if f["descartada"] else f"ok (+{f['reintentos']} reint.)",
               fill=(170, 0, 0) if f["descartada"] else (0, 120, 0))
    d.text((4, lado + 2), Path(str(foto)).name[:24], fill="black")
    return hoja

In [ ]:
%%writefile /kaggle/working/flux_cara/src/ldraw.py
"""Etapa 5: minifig en LDraw con la cara generada aplicada por !TEXMAP.

Todo sale de la librería oficial (complete.zip, se lee sin descomprimir):
    lib = ldraw.Libreria(ldraw.descargar())
    cat = ldraw.catalogo_cabello(lib)                 # estilo -> número de pieza (buscado por descripción)
    rutas = ldraw.escribir_minifig(carpeta, "nombre", rgba, fila, lib, cat)
    ldraw.verificar(rutas, lib) ; ldraw.vista_previa(rutas["ldr"], lib)

La cabeza con cara es un archivo propio (cabeza_<nombre>.dat) armado igual que la cabeza oficial con textura
3626cpx0.dat: s\\3626cs01.dat + la mitad frontal de 3626c.dat dentro de "0 !TEXMAP START PLANAR ..." con
FALLBACK (los visores sin !TEXMAP dibujan la cabeza lisa).
"""
import functools
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import config as C

CRLF = "\r\n"                           # la especificación pide CR/LF (y acepta LF)


# ===================================================================== librería
def _es_libreria(ruta):
    """True si el zip trae ldraw/LDConfig.ldr (la librería oficial completa)."""
    try:
        with zipfile.ZipFile(ruta) as z:
            return any(n.lower().endswith("ldconfig.ldr") for n in z.namelist())
    except (zipfile.BadZipFile, OSError):
        return False


def _buscar_en_inputs():
    """Un complete.zip subido como Input de Kaggle (plan B si la descarga está bloqueada)."""
    for raiz in C.LDRAW_BUSCAR_EN:
        raiz = Path(raiz)
        if not raiz.exists():
            continue
        for r in sorted(raiz.rglob("*.zip")):
            if r.stat().st_size > 10 * 2**20 and _es_libreria(r):
                return r
    return None


def _bajar(url, tmp):
    """urllib con User-Agent explícito; si falla, curl y luego wget. Devuelve el método que funcionó."""
    errores = []
    try:
        req = urllib.request.Request(url, headers={"User-Agent": C.LDRAW_USER_AGENT, "Accept": "*/*"})
        with urllib.request.urlopen(req, timeout=300) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f)
        return "urllib"
    except Exception as e:
        errores.append(f"urllib: {e}")
    for cmd in (["curl", "-fL", "--retry", "3", "-A", C.LDRAW_USER_AGENT, "-o", str(tmp), url],
                ["wget", "-q", "-U", C.LDRAW_USER_AGENT, "-O", str(tmp), url]):
        if shutil.which(cmd[0]) is None:
            errores.append(f"{cmd[0]}: no instalado")
            continue
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode == 0 and tmp.exists() and tmp.stat().st_size > 10 * 2**20:
            return cmd[0]
        errores.append(f"{cmd[0]}: código {p.returncode} {p.stderr.strip()[-200:]}")
    raise RuntimeError("No se pudo descargar la librería LDraw:\n  " + "\n  ".join(errores))


def descargar(destino=None):
    destino = Path(destino or C.LDRAW_ZIP)
    if destino.exists() and destino.stat().st_size > 10 * 2**20:
        print(f"Librería LDraw ya descargada: {destino} ({destino.stat().st_size / 2**20:.0f} MB)")
        return destino
    local = _buscar_en_inputs()
    if local is not None:
        print(f"Librería LDraw tomada de los Inputs: {local} ({local.stat().st_size / 2**20:.0f} MB)")
        return local
    destino.parent.mkdir(parents=True, exist_ok=True)
    t = time.time()
    tmp = destino.with_suffix(".part")
    try:
        metodo = _bajar(C.LDRAW_URL, tmp)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\n\nPlan B: descarga {C.LDRAW_URL} en tu PC, súbelo a Kaggle como Dataset "
            "(Add Input → Upload) y vuelve a correr desde la Etapa 5; se detecta solo en /kaggle/input.") from None
    if not _es_libreria(tmp):
        raise RuntimeError(f"Lo descargado de {C.LDRAW_URL} no es la librería LDraw (sin LDConfig.ldr).")
    tmp.rename(destino)
    print(f"Librería LDraw descargada ({metodo}) de {C.LDRAW_URL}: "
          f"{destino.stat().st_size / 2**20:.0f} MB en {time.time() - t:.0f} s")
    return destino


class Libreria:
    """Acceso de solo lectura a complete.zip (búsqueda: parts/, p/, como dice la especificación)."""

    def __init__(self, ruta_zip):
        self.z = zipfile.ZipFile(ruta_zip)
        self.nombres = {n.lower(): n for n in self.z.namelist()}
        self._desc = None
        lic = self.leer_raiz("LDConfig.ldr")
        self.colores = {m.group(1): (int(m.group(2)), m.group(3))
                        for m in re.finditer(r"!COLOUR\s+(\S+)\s+CODE\s+(\d+)\s+VALUE\s+(#[0-9A-Fa-f]{6})", lic)}
        self.hex_por_codigo = {c: h for c, h in self.colores.values()}
        print(f"Librería: {len(self.nombres)} archivos · {len(self.colores)} colores en LDConfig.ldr")

    def leer_raiz(self, nombre):
        return self.z.read(self.nombres["ldraw/" + nombre.lower()]).decode("utf-8", "replace")

    def ruta(self, nombre):
        n = nombre.replace("\\", "/").lower()
        for base in ("ldraw/parts/", "ldraw/p/"):
            if base + n in self.nombres:
                return self.nombres[base + n]
        return None

    def leer(self, nombre):
        r = self.ruta(nombre)
        return None if r is None else self.z.read(r).decode("utf-8", "replace")

    def textura(self, nombre):
        r = self.nombres.get("ldraw/parts/textures/" + nombre.lower())
        return None if r is None else self.z.read(r)

    def descripciones(self):
        """{pieza.dat: descripción} de las piezas de parts/ (primera línea)."""
        if self._desc is None:
            self._desc = {}
            for n in self.z.namelist():
                if n.startswith("ldraw/parts/") and n.endswith(".dat") and n.count("/") == 2:
                    with self.z.open(n) as h:
                        self._desc[n[12:]] = h.readline().decode("utf-8", "replace")[2:].strip()
        return self._desc

    def por_descripcion(self, texto):
        for pieza, d in self.descripciones().items():
            if d == texto:
                return pieza
        return None

    def codigo(self, nombre_color):
        if nombre_color not in self.colores:
            raise KeyError(f"El color '{nombre_color}' no está en LDConfig.ldr")
        return self.colores[nombre_color][0]


# ===================================================================== catálogo de cabello
def catalogo_cabello(lib):
    filas = []
    for estilo, descs in C.CATALOGO_CABELLO.items():
        pieza = desc = None
        for d in descs:
            pieza = lib.por_descripcion(d)
            if pieza:
                desc = d
                break
        filas.append(dict(estilo=estilo, pieza=pieza, descripcion=desc,
                          buscadas=" | ".join(descs) if descs else "(sin pelo)"))
    cat = pd.DataFrame(filas)
    faltan = cat[(cat["pieza"].isna()) & (cat["buscadas"] != "(sin pelo)")]
    if len(faltan):
        print(f"⚠ Estilos sin pieza en la librería: {faltan['estilo'].tolist()} (revisa config.CATALOGO_CABELLO)")
    total = sum(1 for d in lib.descripciones().values() if d.startswith("Minifig Hair"))
    print(f"Catálogo de cabello: {len(cat)} estilos (de {total} piezas 'Minifig Hair' en la librería)")
    return cat


def elegir_cabello(fila, cat):
    nombre = str(fila.get("nombre"))
    estilo = C.CABELLO_MANUAL.get(nombre)
    fuente = "manual"
    if estilo is None and isinstance(fila.get("cabello"), str) and fila.get("cabello") in C.CATALOGO_CABELLO:
        estilo, fuente = fila.get("cabello"), "dataset"
    if estilo is None:
        fem = fila.get("femenino")
        fem = None if fem is None or (isinstance(fem, float) and np.isnan(fem)) else bool(fem)
        estilo, fuente = C.CABELLO_POR_DEFECTO[fem], "por defecto (femenino=" + str(fem) + ")"
    color_txt = fila.get("color_cejas")
    color_txt = color_txt if isinstance(color_txt, str) else None
    color = C.COLOR_PELO_LDRAW.get(color_txt, C.COLOR_PELO_POR_DEFECTO)
    pieza = cat.set_index("estilo").loc[estilo, "pieza"] if estilo in set(cat["estilo"]) else None
    return dict(estilo=estilo, pieza=pieza if isinstance(pieza, str) else None, color=color,
                fuente=fuente, color_de=color_txt or "defecto")


# ===================================================================== escribir
def _linea1(color, pos, pieza, mat="1 0 0 0 1 0 0 0 1"):
    x, y, z = pos
    return f"1 {color} {x:g} {y:g} {z:g} {mat} {pieza}"


def cabeza_texturizada(lib, nombre_dat, png):
    """Cabeza propia: igual que 3626cpx0.dat pero con nuestra cara y la geometría tomada de 3626c.dat."""
    base = lib.leer(C.CABEZA)
    if base is None:
        raise FileNotFoundError(f"{C.CABEZA} no está en la librería")
    tipo1 = [l.strip() for l in base.splitlines() if l.strip().startswith("1 ")]
    sub = [l for l in tipo1 if l.lower().endswith("3626cs01.dat")]
    frente = [l for l in tipo1 if l not in sub]
    if len(sub) != 1 or not frente:
        raise ValueError(f"{C.CABEZA} no tiene la estructura esperada (s\\3626cs01.dat + frente): {tipo1}")
    texto = [f"0 Minifig Head with Generated Face {nombre_dat[:-4]}", f"0 Name: {nombre_dat}", "0 Author: flux_cara",
             "0 // Basado en 3626c.dat y en la cabeza oficial con textura 3626cpx0.dat", "",
             "0 BFC CERTIFY CCW", "", sub[0], "", "0 // Cara (mitad frontal) con textura",
             f"0 !TEXMAP START PLANAR {C.TEXMAP_PLANAR} {png}"]
    texto += ["0 !: " + l for l in frente]
    texto += ["0 !TEXMAP FALLBACK"] + frente + ["0 !TEXMAP END", ""]
    return CRLF.join(texto)


def lineas_torso(lib, color_torso, color_manos):
    """Líneas de 76382.dat (torso + brazos + manos) con colores propios: manos (3820) aparte."""
    t = lib.leer(C.TORSO)
    if t is None:
        raise FileNotFoundError(f"{C.TORSO} no está en la librería")
    out = []
    for l in t.splitlines():
        p = l.split()
        if p and p[0] == "1" and len(p) >= 15:
            col = color_manos if p[14].lower() == "3820.dat" else color_torso
            out.append(" ".join(["1", str(col)] + p[2:]))
    return out


def escribir_minifig(carpeta, nombre, rgba, fila, lib, cat):
    carpeta = Path(carpeta)
    (carpeta / "textures").mkdir(parents=True, exist_ok=True)
    png, dat, ldr = f"cara_{nombre}.png", f"cabeza_{nombre}.dat", f"minifig_{nombre}.ldr"
    rgba.save(carpeta / "textures" / png)              # la especificación busca textures/<png> y luego <png>
    rgba.save(carpeta / png)
    (carpeta / dat).write_bytes(cabeza_texturizada(lib, dat, png).encode("utf-8"))
    pelo = elegir_cabello(fila, cat)
    c_cab, c_torso = lib.codigo(C.COLOR_CABEZA), lib.codigo(C.COLOR_TORSO)
    lineas = [f"0 Minifig {nombre}", f"0 Name: {ldr}", "0 Author: flux_cara",
              f"0 // Foto: {fila.get('nombre')} - cara FLUX.2-klein-4B + piezas oficiales LDraw",
              f"0 // Cabello: {pelo['estilo']} ({pelo['fuente']}) - color {pelo['color']} (de cejas: {pelo['color_de']})", "",
              "0 // Torso con brazos y manos (lineas de " + C.TORSO + ")"]
    lineas += lineas_torso(lib, c_torso, lib.codigo(C.COLOR_MANOS))
    lineas += ["0 // Caderas y piernas", _linea1(lib.codigo(C.COLOR_PIERNAS), C.POS_PIERNAS, C.PIERNAS),
               "0 // Cabeza con la cara generada (!TEXMAP)", _linea1(c_cab, C.POS_CABEZA, dat)]
    if pelo["pieza"]:
        lineas += ["0 // Cabello", _linea1(lib.codigo(pelo["color"]), C.POS_CABEZA, pelo["pieza"])]
    (carpeta / ldr).write_bytes((CRLF.join(lineas) + CRLF).encode("utf-8"))
    return dict(ldr=carpeta / ldr, dat=carpeta / dat, png=carpeta / "textures" / png, pelo=pelo)


# ===================================================================== verificación
PATRON_TEXMAP = re.compile(r"^0 !TEXMAP START PLANAR( -?\d+(\.\d+)?){9} \S+\.png$")


def verificar(rutas, lib):
    carpeta = rutas["ldr"].parent
    chequeos = []
    faltan = []
    for archivo in (rutas["ldr"], rutas["dat"]):
        for l in archivo.read_text(encoding="utf-8").splitlines():
            l = l[5:] if l.startswith("0 !: ") else l
            p = l.split()
            if p and p[0] == "1":
                pieza = " ".join(p[14:])
                if lib.ruta(pieza) is None and not (carpeta / pieza).exists():
                    faltan.append(pieza)
    chequeos.append(("todas las piezas existen (librería oficial o carpeta)", not faltan, faltan))
    dat = rutas["dat"].read_text(encoding="utf-8").splitlines()
    tex = [l for l in dat if l.startswith("0 !TEXMAP")]
    chequeos.append(("sintaxis START PLANAR (9 números + png)", bool(tex) and bool(PATRON_TEXMAP.match(tex[0])), tex[:1]))
    chequeos.append(("orden START → FALLBACK → END", [t.split()[2] for t in tex] == ["START", "FALLBACK", "END"], tex))
    chequeos.append(("líneas '0 !:' con la geometría texturizada", any(l.startswith("0 !: 1 ") for l in dat), None))
    png = Image.open(rutas["png"])
    chequeos.append(("PNG con alfa en textures/", png.mode == "RGBA" and png.getextrema()[3][0] == 0, png.size))
    chequeos.append(("CR/LF", b"\r\n" in rutas["ldr"].read_bytes(), None))
    for texto, ok, extra in chequeos:
        print(("✅ " if ok else "❌ ") + texto + (f" · {extra}" if extra and not ok else ""))
    return all(ok for _, ok, _ in chequeos)


# ===================================================================== vista previa (aproximada)
class _Escena:
    """Triángulos de un archivo LDraw (recursivo) con color (16 = heredado) y si están texturizados."""

    def __init__(self, lib, carpeta):
        self.lib, self.carpeta = lib, Path(carpeta)
        self.cache = {}

    def _texto(self, nombre):
        local = self.carpeta / nombre
        if local.exists():
            return local.read_text(encoding="utf-8", errors="replace")
        return self.lib.leer(nombre)

    def cargar(self, nombre):
        if nombre in self.cache:
            return self.cache[nombre]
        tris, cols, tex, texdefs = [], [], [], []
        t = self._texto(nombre)
        activo, fallback, planar = False, False, None
        for l in (t or "").splitlines():
            p = l.split()
            if not p:
                continue
            if p[0] == "0":
                if len(p) >= 3 and p[1] == "!TEXMAP":
                    if p[2] == "START" and len(p) >= 13 and p[3] == "PLANAR":
                        activo, fallback = True, False
                        v = np.array(list(map(float, p[4:13]))).reshape(3, 3)
                        texdefs.append((v, " ".join(p[13:])))
                    elif p[2] == "FALLBACK":
                        fallback = True
                    elif p[2] == "END":
                        activo, fallback = False, False
                    continue
                if len(p) > 1 and p[1] == "!:" and activo and not fallback:
                    p = p[2:]
                else:
                    continue
            if fallback:
                continue
            if p[0] == "1" and len(p) >= 15:
                v = list(map(float, p[2:14]))
                M, o = np.array(v[3:]).reshape(3, 3), np.array(v[:3])
                ct, cc, cx, cd = self.cargar(" ".join(p[14:]))
                if len(ct):
                    tris.append(ct @ M.T + o)
                    c = int(p[1])
                    cols.append(np.where(cc == 16, c, cc))
                    tex.append(cx | activo)
                for (tv, png) in cd:
                    texdefs.append((tv @ M.T + o, png))
            elif p[0] in ("3", "4"):
                k = int(p[0])
                pts = np.array(list(map(float, p[2:2 + 3 * k]))).reshape(k, 3)
                caras = [pts[[0, 1, 2]]] + ([pts[[0, 2, 3]]] if k == 4 else [])
                tris.append(np.array(caras))
                cols.append(np.full(len(caras), int(p[1])))
                tex.append(np.full(len(caras), activo))
        r = (np.concatenate(tris) if tris else np.zeros((0, 3, 3)),
             np.concatenate(cols) if cols else np.zeros(0, int),
             np.concatenate(tex) if tex else np.zeros(0, bool), texdefs)
        self.cache[nombre] = r
        return r


def vista_previa(ruta_ldr, lib, lado=None):
    """Vista frontal ortográfica aproximada (no es el render final): sirve para ver que todo encaja."""
    lado = lado or C.LADO_PREVIA
    esc = _Escena(lib, Path(ruta_ldr).parent)
    tris, cols, tex, texdefs = esc.cargar(Path(ruta_ldr).name)
    xy = tris[..., :2]
    mn, mx = xy.reshape(-1, 2).min(0), xy.reshape(-1, 2).max(0)
    s = (lado * 0.9) / (mx - mn).max()
    off = (lado - (mx - mn) * s) / 2 - mn * s
    a = lambda p: p[..., :2] * s + off
    img = Image.new("RGB", (lado, lado), (255, 255, 255))
    masc = Image.new("L", (lado, lado), 0)
    d, dm = ImageDraw.Draw(img), ImageDraw.Draw(masc)
    n = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-9
    luz = np.array([0.3, -0.5, -1.0])
    luz /= np.linalg.norm(luz)
    inten = 0.45 + 0.55 * np.abs(n @ luz)
    orden = np.argsort(-tris[..., 2].mean(1))                 # cámara en -Z: primero lo más lejano
    for i in orden:
        h = lib.hex_por_codigo.get(int(cols[i]), "#808080")
        rgb = tuple(int(int(h[k:k + 2], 16) * inten[i]) for k in (1, 3, 5))
        pol = [tuple(q) for q in a(tris[i])]
        d.polygon(pol, fill=rgb)
        dm.polygon(pol, fill=255 if tex[i] else 0)
    for v, png in texdefs:                                    # PLANAR de frente: la textura es un rectángulo en pantalla
        p1, p2, p3 = a(v)
        x0, x1 = sorted((p1[0], p2[0]))
        y0, y1 = sorted((p1[1], p3[1]))
        ruta = Path(ruta_ldr).parent / "textures" / png
        if not ruta.exists():
            continue
        t = Image.open(ruta).convert("RGBA")
        if p2[0] < p1[0]:
            t = t.transpose(Image.FLIP_LEFT_RIGHT)
        if p3[1] < p1[1]:
            t = t.transpose(Image.FLIP_TOP_BOTTOM)
        w, h = max(1, round(x1 - x0)), max(1, round(y1 - y0))
        t = t.resize((w, h), Image.BOX)
        caja = (round(x0), round(y0))
        m = Image.fromarray((np.asarray(t)[..., 3].astype(float) * np.asarray(masc.crop((*caja, caja[0] + w, caja[1] + h))) / 255).astype(np.uint8))
        img.paste(t.convert("RGB"), caja, m)
    return img


# ===================================================================== README de render
README = """# Render de la minifig ({nombre})

Archivos (deja todos juntos en la misma carpeta):
- `{ldr}`: modelo principal (piezas oficiales de LDraw + la cabeza con cara).
- `{dat}`: cabeza con la cara aplicada con `!TEXMAP START PLANAR` (igual que la cabeza oficial con textura `3626cpx0.dat`).
- `textures/{png}` (y una copia `{png}` al lado): la cara RGBA. El estándar busca primero en `textures/`.
- `vista_previa_{nombre}.png`: vista frontal aproximada hecha en Python, solo para comprobar que las piezas encajan.

Necesitas la librería oficial de piezas: https://library.ldraw.org/library/updates/complete.zip (descomprímela, p. ej. en `C:\\LDraw`).

## LDView
1. Instala LDView. Usa la 4.4 o una más nueva: la 4.4 agregó las texturas CYLINDRICAL/SPHERICAL y arregló varios errores de texturas; la 4.6 y la 4.7 trajeron más arreglos.
2. La primera vez te pide la carpeta de la librería LDraw: elige la carpeta `ldraw` descomprimida.
3. Abre `{ldr}`. Si no ves la cara, revisa que `textures/{png}` esté junto al `.ldr`.
4. Imagen desde la línea de comandos: `LDView {ldr} -SaveSnapshot=render.png` (las demás opciones están en la ayuda de LDView).

## Blender (complemento ImportLDraw de TobyLobster)
1. Instala el complemento desde https://github.com/TobyLobster/ImportLDraw (Blender 4.0 o más nuevo).
   Su README dice que importa las piezas con `!TEXMAP` (planar, cilíndrico y esférico) junto con sus imágenes.
2. *File → Import → LDraw*, elige `{ldr}` y, en las opciones, la carpeta de la librería LDraw.
3. Deja marcada la opción *Import textures*.
4. Renderiza con Cycles o EEVEE.

Visores sin `!TEXMAP` dibujan la cabeza lisa (sección FALLBACK); no es un error del archivo.
"""


def escribir_readme(rutas, nombre):
    texto = README.format(nombre=nombre, ldr=rutas["ldr"].name, dat=rutas["dat"].name, png=rutas["png"].name)
    destino = rutas["ldr"].parent / "README_render.md"
    destino.write_text(texto, encoding="utf-8")
    return destino

In [ ]:
%%writefile /kaggle/working/flux_cara/src/mosaico.py
"""Etapa 6 (Anexo A): misma foto en ambos métodos (modelo propio y FLUX), rondas 1 y 2 y una ronda con Variar.

Una "ronda" es una llamada que genera las 5 caras de una foto, como en la app:
    Ronda 1: rasgos de la foto                 · semillas SEMILLA_FLUX + k   (= Etapa 4, se reutiliza sin regenerar)
    Ronda 2: el usuario pide un cambio          · mismas semillas que la ronda 1, así la única diferencia es el cambio
             (config.CAMBIO_RONDA2, p. ej. {"lentes": "alternar"})
    Variar:  mismo pedido que la ronda 2        · otras semillas, SEMILLA_FLUX + SALTO_VARIAR + k

El modelo propio no vive en este notebook. Sus caras entran de una de dos formas (ver la celda de la Etapa 6):
    GENERAR_PROPIO = función (ruta_foto, rasgos, n, semilla_base) -> lista de n imágenes PIL
                     (en la ronda 2 y en Variar recibe los rasgos ya cambiados)
    config.CARPETA_PROPIO = carpeta con <foto sin extensión>/{ronda1,ronda2,variar}/*.png
Si no hay ninguna, sus filas salen en gris como "pendiente" y el resto del mosaico se arma igual.
"""
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw

import config as C
import cara
import flux
import postproceso
from dataset import cuadrada


def rondas():
    """Las 3 rondas del Anexo A: clave, nombre, si lleva el cambio de la ronda 2 y semilla base."""
    return [dict(clave="ronda1", nombre="Ronda 1", cambio=False, base=C.SEMILLA_FLUX),
            dict(clave="ronda2", nombre="Ronda 2", cambio=True, base=C.SEMILLA_FLUX),
            dict(clave="variar", nombre="Variar", cambio=True, base=C.SEMILLA_FLUX + C.SALTO_VARIAR)]


def aplicar_cambio(fila):
    """Rasgos de la ronda 2: los de la foto con el cambio de config. Devuelve (fila cambiada, texto del cambio).
    "alternar" invierte un rasgo sí/no (lentes, barba, bigote); cualquier otro valor se pone tal cual."""
    cambio = C.CAMBIOS_RONDA2_MANUAL.get(fila["nombre"], C.CAMBIO_RONDA2)
    nueva, textos = fila.copy(), []
    for rasgo, valor in cambio.items():
        antes = fila.get(rasgo)
        if valor == "alternar":
            valor = not bool(flux._si(antes))
            antes = bool(flux._si(antes))
        nueva[rasgo] = valor
        si_no = {True: "si", False: "no"}
        textos.append(f"{rasgo}: {si_no.get(antes, antes)} -> {si_no.get(valor, valor)}")
    return nueva, ", ".join(textos)


def ronda_flux(fila, ronda, plantilla, pipe, marg, tamano, carpeta, previas=None):
    """5 caras de FLUX para una ronda (fila ya trae los rasgos de esa ronda).
    La ronda 1 reutiliza las de la Etapa 4 si existen (mismos rasgos y semillas)."""
    if ronda["clave"] == "ronda1" and previas is not None:
        caras, t = previas
        return caras, t.assign(origen="etapa4 (reutilizada)")
    caras, t = cara.generar_cara(fila["ruta"], fila, n=C.N_CARAS, plantilla=plantilla, pipe=pipe, marg=marg,
                                 tamano=tamano, carpeta=carpeta, semilla_base=ronda["base"])
    return caras, t.assign(origen="generada")


def ronda_propio(fila, ronda, generar=None, carpeta=None):
    """5 caras del modelo propio para una ronda. Devuelve (caras o None, tabla, origen)."""
    n, base = C.N_CARAS, ronda["base"]
    stem = Path(str(fila["nombre"])).stem
    caras, origen = None, "pendiente"
    if generar is not None:
        caras = list(generar(fila["ruta"], fila, n, base))[:n]
        origen = "función GENERAR_PROPIO"
    elif C.CARPETA_PROPIO:
        d = Path(C.CARPETA_PROPIO) / stem / ronda["clave"]
        pngs = sorted(d.glob("*.png"))[:n] if d.is_dir() else []
        if pngs:
            caras = [Image.open(p) for p in pngs]
            origen = f"carpeta {d}"
    if caras is not None:
        caras = [c.convert("RGBA") for c in caras]
        if carpeta is not None:
            carpeta.mkdir(parents=True, exist_ok=True)
            for k, c in enumerate(caras):
                c.save(carpeta / f"cara_{k}.png")
    filas = [dict(k=k, semilla_final=base + k if generar is not None else None, descartada=False,
                  segundos=None, reintentos=0) for k in range(len(caras or []))]
    return caras, pd.DataFrame(filas, columns=["k", "semilla_final", "descartada", "segundos", "reintentos"]), origen


def _celda(c, f, lado):
    """Una cara sobre tablero + pie con semilla y estado."""
    im = Image.new("RGB", (lado, lado + 16), "white")
    p = postproceso.sobre_tablero(c, lado - 8)
    im.paste(p, (4, (lado - p.height) // 2))
    d = ImageDraw.Draw(im)
    s = "" if f is None or pd.isna(f.get("semilla_final")) else f"s{int(f['semilla_final'])}"
    malo = f is not None and bool(f.get("descartada"))
    d.text((4, lado + 2), (s + (" descartada" if malo else "")).strip(), fill=(170, 0, 0) if malo else (60, 60, 60))
    return im


def _pendiente(lado):
    im = Image.new("RGB", (lado, lado + 16), (225, 225, 225))
    d = ImageDraw.Draw(im)
    d.rectangle([2, 2, lado - 3, lado - 3], outline=(170, 170, 170))
    d.text((10, lado // 2 - 6), "pendiente", fill=(110, 110, 110))
    return im


def mosaico(bloques, lado=128, ancho_etiqueta=170):
    """bloques: [dict(foto, nombre, cambio, filas=[dict(metodo, ronda (dict de rondas()), caras o None, tabla, origen)])].
    Por persona: la foto a la izquierda y una fila de 5 caras por método y ronda.
    (Texto sin tildes: la fuente por defecto de PIL no las tiene.)"""
    alto_fila = lado + 16
    n = C.N_CARAS
    ancho = lado + 12 + ancho_etiqueta + n * lado
    alto_bloque = [max(lado + 60, len(b["filas"]) * alto_fila) + 14 for b in bloques]
    hoja = Image.new("RGB", (ancho, 30 + sum(alto_bloque)), "white")
    d = ImageDraw.Draw(hoja)
    r = rondas()
    d.text((6, 4), f"{n} caras por metodo. Ronda 1: rasgos de la foto, semillas {r[0]['base']}+k. "
                   f"Ronda 2: con el cambio pedido, mismas semillas.", fill="black")
    d.text((6, 16), f"Variar: mismo pedido que la ronda 2, semillas {r[2]['base']}+k (k = 0..{n - 1}).", fill="black")
    y = 30
    for b, h in zip(bloques, alto_bloque):
        d.line([(0, y), (ancho, y)], fill=(160, 160, 160))
        hoja.paste(cuadrada(b["foto"], lado), (4, y + 6))
        d.text((4, y + lado + 10), str(b["nombre"])[:22], fill="black")
        d.text((4, y + lado + 24), "cambio ronda 2:", fill=(90, 90, 90))
        d.text((4, y + lado + 38), str(b.get("cambio", ""))[:22], fill=(90, 90, 90))
        for i, fila in enumerate(b["filas"]):
            yy = y + 6 + i * alto_fila
            x0 = lado + 12
            color = (0, 70, 140) if fila["metodo"] == "FLUX" else (140, 60, 0)
            d.text((x0, yy + lado // 2 - 14), fila["metodo"], fill=color)
            d.text((x0, yy + lado // 2), fila["ronda"]["nombre"], fill="black")
            if fila["origen"] == "pendiente":
                d.text((x0, yy + lado // 2 + 14), "pendiente", fill=(150, 150, 150))
            x0 += ancho_etiqueta
            t = fila["tabla"].reset_index(drop=True) if fila["tabla"] is not None else None
            for k in range(n):
                if fila["caras"] is None or k >= len(fila["caras"]):
                    celda = _pendiente(lado)
                else:
                    celda = _celda(fila["caras"][k], t.iloc[k] if t is not None and k < len(t) else None, lado)
                hoja.paste(celda, (x0 + k * lado, yy))
        y += h
    return hoja

In [ ]:
# Importar módulos (se recargan para tomar cambios; el pipeline de FLUX cargado NO se pierde al recargar)
import importlib
if "/kaggle/working/flux_cara/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/flux_cara/src")
import config, registro, dataset, flux, postproceso, ajuste, cara, ldraw, mosaico
for _m in (config, registro, dataset, flux, postproceso, ajuste, cara, ldraw, mosaico):
    importlib.reload(_m)
import pandas as pd
from PIL import Image
registro.iniciar()

---
# Datos: 10 fotos de prueba y márgenes de las estampas reales
Se lee `pares.csv` de PairLegoFace (partición, `pasa`, lentes/barba/bigote; expresión y color de cejas salen de la
descripción de la cara LEGO en el `prompt`). Las estampas reales (`ruta_lego`) se buscan en todos los Inputs.

In [ ]:
registro.seccion("Datos")
SALIDA0 = dataset.carpeta_salida()
RAIZ = dataset.encontrar_dataset()
INV = dataset.inventario(RAIZ)
TABLAS = dataset.leer_tablas(RAIZ)
FOTOS = dataset.tabla_fotos(INV, TABLAS)
ESTAMPAS, INFO_ESTAMPAS = dataset.medir_estampas(FOTOS, INV)
MARG, TAMANO, ORIGEN_MARG = postproceso.margenes(INFO_ESTAMPAS)
print(f"\nEncuadre para la Etapa 3: {ORIGEN_MARG} · {MARG} · tamaño final {TAMANO[0]}x{TAMANO[1]} (ancho x alto)")
ELEGIDAS = dataset.elegir_variadas(FOTOS)
G0 = dataset.grilla(ELEGIDAS)
G0.save(SALIDA0 / "grilla_10.jpg", quality=92)
registro.mostrar(G0)
registro.mostrar(ELEGIDAS[[c for c in ["idx", "nombre", "lentes", "barba", "bigote", "femenino", "edad", "expresion", "color_cejas", "rasgos_lego"] if c in ELEGIDAS]])

---
# Etapa 1 · Cargar FLUX.2-klein-4B y primera prueba

Verificado en la ficha oficial (`black-forest-labs/FLUX.2-klein-4B`, Apache-2.0): el pipeline es `Flux2KleinPipeline`,
es destilado (4 pasos, guidance 1.0) y la foto de referencia se pasa con `image=`. Se carga **una sola vez** con
`DiffusionPipeline.from_pretrained(..., device_map="balanced")` (~16 GB repartidos entre las dos T4).

In [ ]:
registro.seccion("1.1 Cargar el pipeline")
SALIDA1 = flux.carpeta("etapa1")
PIPE = flux.cargar()
FIRMA_OK = flux.verificar_firma(PIPE)

registro.seccion("1.2 Prompts de las 10 fotos")
TABLA1 = flux.prompts(ELEGIDAS)
TABLA1.to_csv(SALIDA1 / "prompts.csv", index=False)
registro.mostrar(TABLA1[["idx", "nombre", "rasgos", "frase_expresion", "frase_lentes", "frase_vello", "frase_color", "por_defecto"]])
print("\nEjemplo de prompt completo (#" + str(TABLA1["idx"].iloc[0]) + "):\n" + TABLA1["prompt"].iloc[0])

In [ ]:
registro.seccion("1.3 Generar (" + str(config.PASOS) + " pasos, guidance " + str(config.GUIA) + ", "
                 + str(config.ANCHO) + "x" + str(config.ALTO) + ", semilla " + str(config.SEMILLA_FLUX) + ")")
RES1 = flux.probar(PIPE, TABLA1, "etapa1", etiqueta="base")
RES1.to_csv(SALIDA1 / "tiempos.csv", index=False)
G1 = flux.grilla_pares(TABLA1, RES1)
G1.save(SALIDA1 / "grilla_etapa1.jpg", quality=92)
registro.mostrar(G1)
registro.seccion("1.4 Autoverificación")
ETAPA1_OK = flux.verificar(TABLA1, RES1, FIRMA_OK)
print("\nETAPA1_OK =", ETAPA1_OK)
registro.empaquetar()

---
# Etapa 2 · Ajuste del prompt
Cada imagen se juzga de forma automática (CLIP zero-shot + color del borde): **plana**, **fondo amarillo**,
**solo rasgos** y **respeta lentes y barba**. Es una estimación: revisa las grillas y corrige con
`config.REVISION_MANUAL` si hace falta. Se detecta la falla más común de la Etapa 1, se prueban las 3 variantes de
`config.VARIANTES[falla]` con las mismas fotos y semilla, y se congela la mejor: primero la que cumple la meta
(≥ 8/10 planas y amarillas), luego **la que más corrige la falla detectada** (para `cara_completa`, más imágenes
con solo rasgos), luego la que más respeta lentes y barba.

In [ ]:
registro.seccion("2.1 Juez sobre la Etapa 1 y falla más común")
SALIDA2 = flux.carpeta("etapa2")
EV = {"base": ajuste.evaluar_lote(TABLA1, RES1, "base")}
RES = {"base": RES1}
registro.mostrar(EV["base"])
FALLA, CUENTAS = ajuste.falla_principal(EV["base"])
FILAS_TP = [ajuste.resumen(EV["base"], config.PROMPT_BASE)]

In [ ]:
registro.seccion("2.2 Tres variantes para la falla '" + FALLA + "'")
PLANTILLAS = {"base": config.PROMPT_BASE, **config.VARIANTES[FALLA]}
for nombre_v, plantilla in config.VARIANTES[FALLA].items():
    print(f"\n--- {nombre_v}")
    t = flux.prompts(ELEGIDAS, plantilla)
    r = flux.probar(PIPE, t, "etapa2", etiqueta=nombre_v)
    RES[nombre_v] = r
    EV[nombre_v] = ajuste.evaluar_lote(t, r, nombre_v)
    FILAS_TP.append(ajuste.resumen(EV[nombre_v], plantilla))
    flux.grilla_pares(t, r).save(SALIDA2 / f"grilla_{nombre_v}.jpg", quality=90)
TABLA_PROMPTS = pd.DataFrame(FILAS_TP)
TABLA_PROMPTS.to_csv(SALIDA2 / "tabla_prompts.csv", index=False)
pd.concat(EV.values()).to_csv(SALIDA2 / "juez_por_imagen.csv", index=False)
registro.mostrar(TABLA_PROMPTS.drop(columns="prompt"))
MEJOR, META_OK = ajuste.elegir(TABLA_PROMPTS, FALLA)
PROMPT_FINAL = PLANTILLAS[MEJOR["variante"]]
(SALIDA2 / "prompt_congelado.txt").write_text(MEJOR["variante"] + "\n" + PROMPT_FINAL, encoding="utf-8")
print("\nPrompt congelado:\n" + PROMPT_FINAL)
registro.empaquetar()

---
# Etapa 3 · Post-proceso a RGBA
Lab → fondo = mediana del borde (5 %) → trazo = distancia > 25 → apertura 3x3 → sin componentes < 15 px →
encuadre con los márgenes medidos en las estampas reales → tamaño final (alfa NEAREST, color BOX).
Guardrail: se descarta si el trazo es < 2 % o > 50 % del área, si menos del 60 % del borde es fondo, o si hay
**cabeza dibujada** (la silueta rellena más grande pasa del 30 % de la imagen: contorno de cabeza, pelo o piel).
Se aplica a las 10 imágenes de la variante congelada.

In [ ]:
registro.seccion("3.1 Post-proceso de las 10 imágenes de '" + MEJOR["variante"] + "'")
SALIDA3 = flux.carpeta("etapa3")
PARES3, ETQ3, FILAS3 = [], [], []
for (_, f), (_, g) in zip(TABLA1.iterrows(), RES[MEJOR["variante"]].iterrows()):
    img = Image.open(g["salida"])
    r = postproceso.a_rgba(img, MARG, TAMANO)
    if r["rgba"] is not None:
        (SALIDA3 / "caras").mkdir(exist_ok=True)
        r["rgba"].save(SALIDA3 / "caras" / f"{int(f['idx']):02d}_{f['nombre'].rsplit('.', 1)[0]}.png")
    PARES3.append((img, r))
    ETQ3.append(f"#{f['idx']} {f['nombre']}")
    FILAS3.append(dict(idx=f["idx"], nombre=f["nombre"], descartada=r["descartada"], motivo=r["motivo"],
                       cobertura=round(r["cobertura"], 3), borde_fondo=round(r["borde_fondo"], 2), caja=r["caja"]))
TABLA3 = pd.DataFrame(FILAS3)
TABLA3.to_csv(SALIDA3 / "postproceso.csv", index=False)
G3 = postproceso.grilla(PARES3, ETQ3)
G3.save(SALIDA3 / "grilla_rgba.jpg", quality=92)
registro.mostrar(G3)
registro.mostrar(TABLA3)
print(f"\nPasan el guardrail: {int((~TABLA3['descartada']).sum())}/{len(TABLA3)}")
registro.empaquetar()

---
# Etapa 4 · Generación completa: `generar_cara(foto, rasgos, n=5)`
Cinco semillas por foto; si el guardrail descarta una, se reintenta con semilla + 1000 (máximo 2 veces).
Se corre para las primeras `config.N_FOTOS_ETAPA4` fotos y se mide el tiempo de las cinco.

In [ ]:
registro.seccion("4.1 generar_cara para " + str(config.N_FOTOS_ETAPA4) + " fotos")
SALIDA4 = flux.carpeta("etapa4")
CARAS4, TABLAS4 = {}, []
for _, f in ELEGIDAS.head(config.N_FOTOS_ETAPA4).iterrows():
    nombre = f["nombre"].rsplit(".", 1)[0]
    caras, t = cara.generar_cara(f["ruta"], f, n=config.N_CARAS, plantilla=PROMPT_FINAL, pipe=PIPE,
                                 marg=MARG, tamano=TAMANO, carpeta=SALIDA4 / nombre)
    print(f"{f['nombre']}: {len(caras)} caras en {t.attrs['segundos_total']:.1f} s · reintentos {int(t['reintentos'].sum())}"
          f" · descartadas {int(t['descartada'].sum())}")
    CARAS4[f["nombre"]] = (f, caras, t)
    TABLAS4.append(t.assign(foto=f["nombre"]))
    tira = cara.tira(f["ruta"], caras, t)
    tira.save(SALIDA4 / f"tira_{nombre}.jpg", quality=92)
    registro.mostrar(tira)
TABLA4 = pd.concat(TABLAS4, ignore_index=True)
TABLA4.to_csv(SALIDA4 / "generacion.csv", index=False)
registro.mostrar(TABLA4[["foto", "k", "semilla_final", "reintentos", "descartada", "segundos", "intentos"]])
print(f"\nTiempo medio por cara (con reintentos): {TABLA4['segundos'].mean():.1f} s")
registro.empaquetar()

---
# Etapa 5 · Ensamblado en LDraw
Se descarga la librería oficial (library.ldraw.org, `complete.zip`) y se lee sin descomprimir.
- Cabeza `3626c.dat` con la cara por `!TEXMAP START PLANAR -13 0 10 13 0 10 -13 21 10` (la misma convención que la
  cabeza oficial con textura `3626cpx0.dat`), con FALLBACK para visores sin texturas.
- Torso con brazos y manos `76382.dat`, caderas y piernas `73200b-f1.dat`, cabello del catálogo (buscado por
  descripción en la librería). PairLegoFace no trae cabello: el estilo sale de `config.CABELLO_MANUAL` o, si no, de
  `femenino` (supuesto), y el color, de las cejas.
- Cara de la minifig: de las 5 de la Etapa 4 (sin las descartadas), la que más rasgos de la foto respeta según CLIP
  (lentes y vello facial = barba o bigote, umbral `config.UMBRAL_RASGO_CLIP`); a igual puntaje, la de menor k.
  Los puntajes quedan en `eleccion_cara.csv`.
- Por cada foto de la Etapa 4: `minifig_<foto>.ldr`, `cabeza_<foto>.dat`, `textures/cara_<foto>.png`,
  `README_render.md` y una vista previa frontal aproximada.

In [ ]:
registro.seccion("5.1 Librería oficial y catálogo de cabello")
SALIDA5 = flux.carpeta("etapa5")
LIB = ldraw.Libreria(ldraw.descargar())
CATALOGO = ldraw.catalogo_cabello(LIB)
CATALOGO.to_csv(SALIDA5 / "catalogo_cabello.csv", index=False)
registro.mostrar(CATALOGO)

registro.seccion("5.2 Minifigs")
ETAPA5_OK = True
for foto, (f, caras, t) in CARAS4.items():
    nombre = foto.rsplit(".", 1)[0]
    rgba, i, puntos = cara.mejor(caras, t, f)
    rutas = ldraw.escribir_minifig(SALIDA5 / nombre, nombre, rgba, f, LIB, CATALOGO)
    print(f"\n{foto}: cara #{i} · cabello {rutas['pelo']}")
    if puntos is not None:
        puntos.to_csv(SALIDA5 / nombre / "eleccion_cara.csv", index=False)
        registro.mostrar(puntos)
    ETAPA5_OK &= ldraw.verificar(rutas, LIB)
    ldraw.escribir_readme(rutas, nombre)
    previa = ldraw.vista_previa(rutas["ldr"], LIB)
    previa.save(SALIDA5 / nombre / f"vista_previa_{nombre}.png")
    registro.mostrar(previa)
print("\nETAPA5_OK =", ETAPA5_OK)

---
# Etapa 6 · Anexo A: misma foto en ambos métodos, rondas 1 y 2 y una ronda con Variar
**Ronda** = una llamada que genera las 5 caras de una foto, como en la app.

| Ronda | Rasgos | Semillas |
|---|---|---|
| **Ronda 1** | los de la foto | `SEMILLA_FLUX + k` (42–46). En FLUX son las caras de la Etapa 4: se reutilizan |
| **Ronda 2** | los de la foto **con el cambio que pide el usuario** (`config.CAMBIO_RONDA2`) | las mismas de la ronda 1: la única diferencia es el cambio |
| **Variar** | el mismo pedido de la ronda 2 | otras: `SEMILLA_FLUX + SALTO_VARIAR + k` (142–146) |

El cambio por defecto es `{"lentes": "alternar"}`: si la persona tiene lentes se le quitan y si no, se le ponen.
Se puede pedir otro (`{"expresion": "sorpresa"}`, `{"barba": True}`, …) o uno por foto en `config.CAMBIOS_RONDA2_MANUAL`.

Para cada una de las `config.N_PERSONAS_MOSAICO` personas, el mosaico muestra la foto y 6 filas de 5 caras
(modelo propio y FLUX × ronda 1, ronda 2 y Variar). Cada cara lleva su semilla; las descartadas por el guardrail
salen en rojo. Para el Anexo A basta con el mosaico de una persona: `etapa6/mosaico_<foto>.jpg`.

**Modelo propio.** No vive en este notebook. Sus caras entran de una de estas dos formas (la primera que exista):
1. En la celda siguiente, `GENERAR_PROPIO = función(ruta_foto, rasgos, n, semilla_base)` que devuelva `n` imágenes PIL.
   En la ronda 2 y en Variar recibe los rasgos ya cambiados.
2. `config.CARPETA_PROPIO` apuntando a un Input con `<foto sin extensión>/ronda1/`, `/ronda2/` y `/variar/` (PNG).

Si no hay ninguna, las filas del modelo propio salen en gris como **pendiente** y el resto se arma igual.

In [ ]:
# Modelo propio: reemplaza None por la función de Persona 4, por ejemplo
#   def GENERAR_PROPIO(ruta_foto, rasgos, n, semilla_base):
#       return [modelo_propio(ruta_foto, rasgos, semilla=semilla_base + k) for k in range(n)]
GENERAR_PROPIO = None

registro.seccion("6.1 Anexo A · " + str(config.N_PERSONAS_MOSAICO) + " personas · " + config.NOMBRE_PROPIO
                 + " y FLUX · ronda 1, ronda 2 y Variar")
SALIDA6 = flux.carpeta("etapa6")
BLOQUES6, FILAS6 = [], []
for _, f in ELEGIDAS.head(config.N_PERSONAS_MOSAICO).iterrows():
    nombre = f["nombre"].rsplit(".", 1)[0]
    f2, cambio = mosaico.aplicar_cambio(f)
    print(f"\n{f['nombre']} · cambio pedido en la ronda 2: {cambio}")
    print("  prompt ronda 1:", flux.frases(f)[0])
    print("  prompt ronda 2:", flux.frases(f2)[0])
    filas = []
    for metodo in (config.NOMBRE_PROPIO, "FLUX"):
        for r in mosaico.rondas():
            fr = f2 if r["cambio"] else f
            carpeta_r = SALIDA6 / nombre / f"{'flux' if metodo == 'FLUX' else 'propio'}_{r['clave']}"
            if metodo == "FLUX":
                previas = CARAS4[f["nombre"]][1:] if f["nombre"] in CARAS4 else None
                caras, t = mosaico.ronda_flux(fr, r, PROMPT_FINAL, PIPE, MARG, TAMANO, carpeta_r, previas)
                origen = t["origen"].iloc[0]
            else:
                caras, t, origen = mosaico.ronda_propio(fr, r, GENERAR_PROPIO, carpeta_r)
            filas.append(dict(metodo=metodo, ronda=r, caras=caras, tabla=t, origen=origen))
            n_ok = 0 if caras is None else int((~t["descartada"].astype(bool)).sum())
            estado = "pendiente" if caras is None else f"{n_ok}/{len(caras)} ok"
            print(f"  {metodo:<14} {r['nombre']:<8}: {estado} · semillas {r['base']}+k · {origen}")
            comun = dict(foto=f["nombre"], metodo=metodo, ronda=r["clave"], cambio=cambio if r["cambio"] else "",
                         origen=origen, prompt=PROMPT_FINAL.format(**flux.frases(fr)[0]))
            for _, x in t.iterrows():
                FILAS6.append(dict(**comun, k=x["k"], semilla=x["semilla_final"], descartada=bool(x["descartada"]),
                                   segundos=x["segundos"]))
            if caras is None:
                FILAS6.append(comun)
    BLOQUES6.append(dict(foto=f["ruta"], nombre=f["nombre"], cambio=cambio, filas=filas))
    mosaico.mosaico([BLOQUES6[-1]]).save(SALIDA6 / f"mosaico_{nombre}.jpg", quality=92)
TABLA6 = pd.DataFrame(FILAS6)
TABLA6.to_csv(SALIDA6 / "rondas.csv", index=False)
M6 = mosaico.mosaico(BLOQUES6)
M6.save(SALIDA6 / "mosaico_rondas.jpg", quality=92)
registro.mostrar(M6)
PROPIO_LISTO = not (TABLA6["origen"] == "pendiente").any()
print("\nMosaico:", SALIDA6 / "mosaico_rondas.jpg", "· modelo propio", "incluido" if PROPIO_LISTO else "PENDIENTE (filas en gris)")
registro.empaquetar()

## Resumen y paquete final
Descarga **`flux_cara/entregar.zip`** del panel *Output* y mándalo.

In [ ]:
registro.seccion("Resumen")
print("Etapa 1 OK:", ETAPA1_OK)
print("Etapa 2: falla más común", repr(FALLA), CUENTAS, "· congelada", repr(MEJOR["variante"]), "· meta 8/10:", META_OK)
print("Etapa 3: pasan el guardrail", int((~TABLA3["descartada"]).sum()), "de", len(TABLA3))
print("Etapa 4: caras", len(TABLA4), "· descartadas", int(TABLA4["descartada"].sum()), "· s/cara", round(TABLA4["segundos"].mean(), 1))
print("Etapa 5 OK:", ETAPA5_OK)
print("Etapa 6 (Anexo A): mosaico de", len(BLOQUES6), "personas · ronda 1, ronda 2 con cambio y Variar · modelo propio",
      "incluido" if PROPIO_LISTO else "pendiente")
registro.empaquetar()
registro.detener()